# Infosys Springboard Internship 7.0 — Batch 4
## Milestone 1: User Authentication Module (Google Colab Submission Deliverable)

### Overview & Features Implemented
This notebook delivers the thoroughly validated and tested implementation of **Milestone 1**:
1. **Strict Password Policy**: $\ge$8 characters, uppercase, lowercase, digit, special character, whitespace policy, Unicode safe.
2. **Email & Username Validation**: RFC 5322 compliance, 3-point min-length rules ($\ge$2 before `@`, $\ge$2 between `@` and dot, $\ge$2 after final dot), case-insensitive uniqueness.
3. **Email Verification via Gmail SMTP**: Account activation OTP codes (10-min expiry, max 3 attempts, single-use). Unverified accounts cannot log in.
4. **Dual Login & Account Lockout**: Login via Username or Email; 5 failed login attempts trigger a 15-minute lockout with automatic expiry.
5. **Separate Admin Authentication**: Dedicated admin login and dashboard displaying registered users without exposing hashes or credentials.
6. **Password Recovery (Two Methods)**:
   - **Method 1**: Security Question verification with bcrypt-hashed answers.
   - **Method 2**: Gmail One-Time Password (OTP) with 5-minute expiry, max 3 attempts, single-use enforcement, and 60s resend cooldown.
7. **JWT Session Management**: Signed access tokens, validation before dashboard access, automatic logout cleanup.
8. **Automated Unit & Integration Testing**: 77 automated pytest tests executing against isolated test databases.


In [ ]:
# ------------------------------------------------------------------------------
# 1. Install Required Dependencies
# ------------------------------------------------------------------------------
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly pytest


In [ ]:
# ------------------------------------------------------------------------------
# 2. Safe Google Colab Secrets Verification & Environment Forwarding
# ------------------------------------------------------------------------------
import os
secret_names = ['JWT_SECRET', 'NGROK_AUTHTOKEN', 'EMAIL_PASSWORD', 'EMAIL_ADDRESS']
print('=' * 65)
print('🔐 Checking Google Colab Secrets Configuration:')
print('=' * 65)

for name in secret_names:
    is_present = False
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val and len(str(val).strip()) > 0:
            is_present = True
            # Safely forward secret to process environment for child processes (Streamlit & Pytest)
            os.environ[name] = str(val).strip()
    except Exception:
        val = os.environ.get(name)
        is_present = bool(val and len(str(val).strip()) > 0)
    status_symbol = '✅ Present & Forwarded' if is_present else '⚠️ Missing / Not Configured'
    print(f'  • {name:<20}: {status_symbol}')

print('=' * 65)
print('💡 Setup Notes for Colab Secrets (Left sidebar Key icon):')
print('   1. Notebook Access: Ensure the toggle is switched ON for all 4 secrets.')
print('   2. EMAIL_PASSWORD: Use a 16-character Google App Password from:')
print('      Google Account > Security > 2-Step Verification > App passwords.')
print('      (Any spaces in the App Password are automatically handled and trimmed).')
print('   3. EMAIL_ADDRESS: Your full Gmail address (e.g., example@gmail.com).')
print('   4. NGROK_AUTHTOKEN: Your token from https://dashboard.ngrok.com/get-started/your-authtoken')
print('   5. JWT_SECRET: Any strong secret key for HS256 token signing.')
print('=' * 65)


In [ ]:
%%writefile auth_utils.py
"""
auth_utils.py - Core Authentication & Validation Utilities
Infosys Springboard Internship 7.0 (Batch 4) - Milestone 1
Complete Implementation:
- Password validation with live criteria checks
- 3-point RFC email validation and username constraints
- Parameterized SQLite with case-insensitive uniqueness
- Bcrypt password & security answer hashing
- Email verification with Gmail SMTP, OTP expiration, and unverified login prevention
- Dual-channel password recovery (Security Question & Gmail OTP)
- Account lockout policy (5 failed attempts = 15-minute lock)
- Dedicated admin authentication and sanitized directory
- JWT session management with zero hardcoded fallback secrets
"""

import os
import re
import secrets
import smtplib
import sqlite3
import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from typing import Tuple, Optional, Dict, Any, List
import bcrypt
import jwt

# ==============================================================================
# CONFIGURATION & CONSTANTS
# ==============================================================================
APPROVED_SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your mother's maiden name?",
    "What is your favourite city?",
]

DEFAULT_DB_PATH = os.environ.get("DATABASE_PATH", "infosys_portal.db")

# Account Lockout Policy
MAX_FAILED_LOGIN_ATTEMPTS = 5
LOCKOUT_DURATION_MINUTES = 15

# OTP & Email Verification Policy
OTP_EXPIRY_MINUTES = 5
VERIFICATION_EXPIRY_MINUTES = 10
OTP_MAX_VERIFY_ATTEMPTS = 3
OTP_RESEND_COOLDOWN_SECONDS = 60

JWT_ALGORITHM = "HS256"


def get_jwt_secret() -> str:
    """
    Retrieves the JWT secret from Colab Secrets (via userdata) or environment.
    Fails with a clear ValueError if the secret is missing.
    Never returns a hardcoded production fallback secret.
    """
    # 1. Google Colab Secrets
    try:
        from google.colab import userdata
        sec = userdata.get("JWT_SECRET")
        if sec and len(str(sec).strip()) > 0:
            return str(sec).strip()
    except Exception:
        pass

    # 2. Environment Variable
    sec = os.environ.get("JWT_SECRET")
    if sec and len(str(sec).strip()) > 0:
        return str(sec).strip()

    raise ValueError(
        "CRITICAL CONFIGURATION ERROR: 'JWT_SECRET' is missing. "
        "Please configure 'JWT_SECRET' in Google Colab Secrets (with Notebook access enabled) "
        "or set the 'JWT_SECRET' environment variable for local execution."
    )


# ==============================================================================
# 1. LIVE PASSWORD CRITERIA & POLICY
# ==============================================================================
def check_password_criteria(password: str) -> Dict[str, bool]:
    """
    Evaluates password against individual security policy criteria for live UI checklists:
    - min_length: >= 8 characters
    - uppercase: at least 1 uppercase letter (A-Z)
    - lowercase: at least 1 lowercase letter (a-z)
    - digit: at least 1 number (0-9)
    - special: at least 1 special character
    - all_passed: True if all 5 criteria are met
    """
    if not password:
        return {
            "min_length": False,
            "uppercase": False,
            "lowercase": False,
            "digit": False,
            "special": False,
            "all_passed": False,
        }

    c_len = len(password) >= 8
    c_upper = bool(re.search(r"[A-Z]", password))
    c_lower = bool(re.search(r"[a-z]", password))
    c_digit = bool(re.search(r"[0-9]", password))
    c_special = bool(re.search(r"""[!@#$%^&*()_+\-=\[\]{};':"\\|,.<>\/?`~]""", password))
    all_ok = c_len and c_upper and c_lower and c_digit and c_special and len(password.strip()) > 0

    return {
        "min_length": c_len,
        "uppercase": c_upper,
        "lowercase": c_lower,
        "digit": c_digit,
        "special": c_special,
        "all_passed": all_ok,
    }


def validate_password(password: str) -> Tuple[bool, str]:
    """
    Validates a password according to the Infosys Milestone 1 security policy:
    - At least 8 characters
    - At least one uppercase letter (A-Z)
    - At least one lowercase letter (a-z)
    - At least one digit (0-9)
    - At least one special character (!@#$%^&*()_+-=[]{};':"|,.<>/?`~)
    - Cannot be empty or whitespace-only
    - Whitespace policy: Leading/trailing and embedded spaces are permitted
      if other criteria are satisfied, and must never be silently trimmed or modified.
    - Unicode and unusual characters are handled safely without crashing.
    """
    if not password or len(password.strip()) == 0:
        return False, "Password cannot be empty or whitespace only."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least one uppercase letter (A-Z)."

    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least one lowercase letter (a-z)."

    if not re.search(r"[0-9]", password):
        return False, "Password must contain at least one digit (0-9)."

    if not re.search(r"""[!@#$%^&*()_+\-=\[\]{};':"\\|,.<>\/?`~]""", password):
        return False, "Password must contain at least one special character (!@#$%^&* etc.)."

    return True, ""


# ==============================================================================
# 2. EMAIL VALIDATION & NORMALIZATION
# ==============================================================================
def normalize_email(email: str) -> str:
    """Normalizes email to lowercase with stripped leading/trailing spaces."""
    if not email:
        return ""
    return email.strip().lower()


def validate_email(email: str) -> Tuple[bool, str]:
    """
    Validates email according to RFC standards and milestone explicit rules:
    - Non-empty, no internal spaces
    - Exactly one '@' symbol
    - At least two characters before '@' (local part)
    - At least two characters between '@' and the dot (domain name)
    - At least two characters after the final dot (TLD)
    """
    if not email or len(email.strip()) == 0:
        return False, "Email address cannot be empty."

    email = email.strip()

    if " " in email:
        return False, "Email address cannot contain spaces."

    if email.count("@") != 1:
        return False, "Email address must contain exactly one '@' symbol."

    local_part, domain_part = email.split("@")

    if len(local_part) < 2:
        return False, "Email local part before '@' must contain at least 2 characters."

    if not domain_part:
        return False, "Email address is missing the domain part after '@'."

    if "." not in domain_part:
        return False, "Email domain must contain at least one dot separating domain and TLD."

    domain_segments = domain_part.split(".")
    domain_name = domain_segments[0]
    tld = domain_segments[-1]

    if len(domain_name) < 2:
        return False, "Email domain name between '@' and dot must contain at least 2 characters."

    if len(tld) < 2 or not tld.isalpha():
        return False, "Email domain must end with a valid Top-Level Domain of at least 2 letters (e.g., .com, .ai, .org)."

    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)+$"
    if not re.match(pattern, email):
        return False, "Invalid email address format."

    return True, ""


# ==============================================================================
# 3. USERNAME VALIDATION & NORMALIZATION
# ==============================================================================
def normalize_username(username: str) -> str:
    """Normalizes username by stripping leading/trailing whitespace."""
    if not username:
        return ""
    return username.strip()


def validate_username(username: str) -> Tuple[bool, str]:
    """
    Validates username:
    - Between 3 and 50 characters
    - Cannot be empty or whitespace-only
    - Allowed characters: alphanumeric, spaces, hyphens, dots, underscores
    """
    if not username or len(username.strip()) == 0:
        return False, "Username cannot be empty or whitespace only."

    norm_user = username.strip()
    if len(norm_user) < 3:
        return False, "Username must be at least 3 characters long."

    if len(norm_user) > 50:
        return False, "Username cannot exceed 50 characters."

    if not re.match(r"^[a-zA-Z0-9_.\- ]+$", norm_user):
        return False, "Username can only contain letters, numbers, spaces, hyphens, dots, and underscores."

    return True, ""


# ==============================================================================
# 4. SECURITY QUESTION & ANSWER VALIDATION
# ==============================================================================
def normalize_security_answer(answer: str) -> str:
    """Normalizes security answer: strips whitespace and converts to lowercase."""
    if not answer:
        return ""
    return answer.strip().lower()


def validate_security_question(question: str) -> Tuple[bool, str]:
    """Validates that the security question is from the approved list."""
    if question not in APPROVED_SECURITY_QUESTIONS:
        return False, "Please select an approved security question from the list."
    return True, ""


def validate_security_answer(answer: str) -> Tuple[bool, str]:
    """Validates that the security answer is non-empty and non-whitespace."""
    if not answer or len(answer.strip()) == 0:
        return False, "Security answer cannot be empty or whitespace only."
    if len(answer.strip()) < 2:
        return False, "Security answer must be at least 2 characters long."
    return True, ""


# ==============================================================================
# 5. CRYPTOGRAPHIC HASHING (BCRYPT)
# ==============================================================================
def hash_text(plaintext: str) -> str:
    """Hashes a text string using bcrypt with a secure salt."""
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(plaintext.encode("utf-8"), salt).decode("utf-8")


def check_hash(plaintext: str, hashed: Optional[str]) -> bool:
    """Verifies a plaintext string against a bcrypt hash."""
    if not plaintext or not hashed:
        return False
    try:
        return bcrypt.checkpw(plaintext.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False


# ==============================================================================
# 6. DATABASE SCHEMA & OPERATIONS
# ==============================================================================
def get_db(db_path: Optional[str] = None) -> sqlite3.Connection:
    """Returns a SQLite connection with row factory enabled."""
    path = db_path or DEFAULT_DB_PATH
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db(db_path: Optional[str] = None, seed_admin: bool = True) -> None:
    """Initializes the database schema with support for email verification."""
    with get_db(db_path) as conn:
        cursor = conn.cursor()

        # 1. Users Table (with is_verified column)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                account_status TEXT DEFAULT 'ACTIVE',
                failed_attempts INTEGER DEFAULT 0,
                locked_until TEXT DEFAULT NULL,
                is_verified INTEGER DEFAULT 0,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)
        # Backward-compatible column migration
        try:
            cursor.execute("ALTER TABLE users ADD COLUMN is_verified INTEGER DEFAULT 0")
        except sqlite3.OperationalError:
            pass

        cursor.execute("""
            CREATE UNIQUE INDEX IF NOT EXISTS idx_users_username_lower
            ON users(LOWER(username))
        """)
        cursor.execute("""
            CREATE UNIQUE INDEX IF NOT EXISTS idx_users_email_lower
            ON users(LOWER(email))
        """)

        # 2. Email Verification Table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS email_verifications (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                code_hash TEXT NOT NULL,
                attempts INTEGER DEFAULT 0,
                expires_at TEXT NOT NULL,
                used INTEGER DEFAULT 0,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # 3. Dedicated Admin Users Table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS admin_users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                role TEXT DEFAULT 'ADMIN',
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # 4. OTP Recovery Table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS otp_records (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                otp_hash TEXT NOT NULL,
                attempts INTEGER DEFAULT 0,
                expires_at TEXT NOT NULL,
                used INTEGER DEFAULT 0,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)

        if seed_admin:
            cursor.execute("SELECT id FROM admin_users WHERE LOWER(email) = 'infosys@ai'")
            if not cursor.fetchone():
                admin_user = "Administrator"
                admin_email = "infosys@ai"
                admin_pwd = hash_text("Admin@123")
                cursor.execute(
                    """
                    INSERT INTO admin_users (username, email, password_hash, role)
                    VALUES (?, ?, ?, 'ADMIN')
                    """,
                    (admin_user, admin_email, admin_pwd),
                )
        conn.commit()


# ==============================================================================
# 7. EMAIL VERIFICATION WORKFLOW (SMTP GMAIL)
# ==============================================================================
def generate_code_6digit() -> str:
    """Generate a secure, unpredictable six-digit OTP."""
    return f"{secrets.randbelow(900000) + 100000}"


def create_email_verification(email: str, db_path: Optional[str] = None) -> Tuple[bool, Optional[str], str]:
    """
    Generates a 6-digit email verification code, bcrypt-hashes it, and records it with 10-minute expiry.
    Returns (True, plaintext_code_for_sender, message).
    """
    clean_email = normalize_email(email)
    now_utc = datetime.datetime.utcnow()

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()

            # Invalidate older unverified codes for this email
            cursor.execute("UPDATE email_verifications SET used = 2 WHERE LOWER(email) = LOWER(?) AND used = 0", (clean_email,))

            code = generate_code_6digit()
            code_h = hash_text(code)
            exp = (now_utc + datetime.timedelta(minutes=VERIFICATION_EXPIRY_MINUTES)).isoformat()
            created_at = now_utc.isoformat()

            cursor.execute(
                """
                INSERT INTO email_verifications (email, code_hash, attempts, expires_at, used, created_at)
                VALUES (?, ?, 0, ?, 0, ?)
                """,
                (clean_email, code_h, exp, created_at),
            )
            conn.commit()
            return True, code, "Verification code generated."
    except Exception as e:
        return False, None, f"Database error creating verification: {str(e)}"


def get_email_credentials() -> Tuple[Optional[str], Optional[str]]:
    """
    Safely retrieves EMAIL_ADDRESS and EMAIL_PASSWORD from Google Colab userdata or os.environ.
    Automatically trims whitespace and removes spaces from 16-character Google 2-Step App Passwords.
    """
    user = None
    pwd = None
    try:
        from google.colab import userdata
        user = userdata.get("EMAIL_ADDRESS")
        pwd = userdata.get("EMAIL_PASSWORD")
    except Exception:
        pass

    if not user:
        user = os.environ.get("EMAIL_ADDRESS")
    if not pwd:
        pwd = os.environ.get("EMAIL_PASSWORD")

    u = str(user).strip() if user and len(str(user).strip()) > 0 else None
    p = str(pwd).strip().replace(" ", "") if pwd and len(str(pwd).strip()) > 0 else None
    return u, p


def _dispatch_gmail_smtp(
    to_email: str,
    subject: str,
    body_text: str,
    smtp_user: Optional[str] = None,
    smtp_pass: Optional[str] = None,
) -> Tuple[bool, str]:
    """
    Dispatches an email message via Gmail SMTP with dual SSL (465) / TLS (587) fallback.
    Sanitizes credentials, stripping spaces from Google 2-Step Verification App Passwords.
    """
    if smtp_user is None or smtp_pass is None:
        auto_u, auto_p = get_email_credentials()
        if smtp_user is None:
            smtp_user = auto_u
        if smtp_pass is None:
            smtp_pass = auto_p

    if not smtp_user or not smtp_pass or len(str(smtp_user).strip()) == 0 or len(str(smtp_pass).strip()) == 0:
        return False, "SMTP credentials missing. Please configure EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets."

    clean_user = str(smtp_user).strip()
    clean_pass = str(smtp_pass).strip().replace(" ", "")

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = f"Infosys Security Portal <{clean_user}>"
    msg["To"] = to_email
    msg.attach(MIMEText(body_text, "plain"))

    # Attempt 1: Port 465 (SSL)
    ssl_err = None
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=12) as server:
            server.login(clean_user, clean_pass)
            server.sendmail(clean_user, to_email, msg.as_string())
        return True, "Email successfully sent via Gmail SMTP SSL."
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail SMTP Authentication failed. Please verify your 16-character Google App Password (generated via Google Account > Security > 2-Step Verification > App passwords)."
    except Exception as e:
        ssl_err = e

    # Attempt 2: Port 587 (STARTTLS fallback for Colab/restricted network environments)
    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=12) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(clean_user, clean_pass)
            server.sendmail(clean_user, to_email, msg.as_string())
        return True, "Email successfully sent via Gmail SMTP TLS."
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail SMTP Authentication failed. Please verify your 16-character Google App Password (generated via Google Account > Security > 2-Step Verification > App passwords)."
    except Exception as tls_err:
        return False, f"Failed to deliver email via Gmail SMTP: {str(ssl_err or tls_err)}"


def send_verification_email(
    to_email: str,
    code: str,
    smtp_user: Optional[str] = None,
    smtp_pass: Optional[str] = None,
) -> Tuple[bool, str]:
    """
    Sends the verification code using Gmail SMTP (SSL 465 / TLS 587 fallback).
    Fails with clear error if credentials missing or delivery rejected.
    """
    subject = "Verify Your Infosys Portal Account"
    body = f"""Hello,

Welcome to the Infosys Intelligent Analytics Platform!

Your 6-digit account verification code is:

{code}

This code is valid for {VERIFICATION_EXPIRY_MINUTES} minutes and required to activate your account.
If you did not request this, please disregard this email.

Best regards,
Infosys Security Operations
"""
    return _dispatch_gmail_smtp(to_email, subject, body, smtp_user, smtp_pass)


def verify_email_account(email: str, code: str, db_path: Optional[str] = None) -> Tuple[bool, str]:
    """
    Verifies user email using the submitted code:
    - Checks expiry (10 mins)
    - Limits attempts to max 3
    - Enforces single-use
    - On success: marks user as verified (is_verified = 1)
    """
    if not email or not code:
        return False, "Email address and verification code are required."

    clean_email = normalize_email(email)
    clean_code = code.strip()

    if not clean_code.isdigit() or len(clean_code) != 6:
        return False, "Verification code must be a 6-digit number."

    now_utc = datetime.datetime.utcnow()

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT id, code_hash, attempts, expires_at, used
                FROM email_verifications
                WHERE LOWER(email) = LOWER(?) AND used = 0
                ORDER BY id DESC LIMIT 1
                """,
                (clean_email,),
            )
            rec = cursor.fetchone()
            if not rec:
                return False, "No active verification request found for this email. Please request a new code."

            v_id = rec["id"]
            code_h = rec["code_hash"]
            attempts = rec["attempts"]
            expires_at_str = rec["expires_at"]

            if attempts >= OTP_MAX_VERIFY_ATTEMPTS:
                cursor.execute("UPDATE email_verifications SET used = 2 WHERE id = ?", (v_id,))
                conn.commit()
                return False, "Maximum verification attempts exceeded. Please request a new code."

            try:
                expires_at_dt = datetime.datetime.fromisoformat(expires_at_str)
                if now_utc > expires_at_dt:
                    cursor.execute("UPDATE email_verifications SET used = 2 WHERE id = ?", (v_id,))
                    conn.commit()
                    return False, "Verification code has expired. Please request a new code."
            except Exception:
                return False, "Invalid verification timestamp."

            if not check_hash(clean_code, code_h):
                cursor.execute("UPDATE email_verifications SET attempts = attempts + 1 WHERE id = ?", (v_id,))
                conn.commit()
                rem = OTP_MAX_VERIFY_ATTEMPTS - (attempts + 1)
                return False, f"Incorrect verification code. {rem} attempt(s) remaining."

            # Mark code as used
            cursor.execute("UPDATE email_verifications SET used = 1 WHERE id = ?", (v_id,))

            # Activate and verify account
            cursor.execute("UPDATE users SET is_verified = 1, account_status = 'ACTIVE' WHERE LOWER(email) = LOWER(?)", (clean_email,))
            conn.commit()
            return True, "Email verified successfully! Your account is now active. Please sign in."

    except Exception as e:
        return False, f"Verification error: {str(e)}"


# ==============================================================================
# 8. SIGNUP REGISTRATION FLOW
# ==============================================================================
def register_user(
    username: str,
    email: str,
    password: str,
    confirm_password: str,
    security_question: str,
    security_answer: str,
    db_path: Optional[str] = None,
    auto_verify: bool = False,
) -> Tuple[bool, str]:
    """
    Validates all mandatory fields server-side and registers a new user atomically.
    Ensures:
    - Mandatory field presence
    - Username & Email policies
    - Full 5-point password criteria
    - Exact confirmation match
    - Approved security question & bcrypt hashed answer
    - is_verified set to 0 by default (requires verification before login)
    - Rejects duplicate usernames & emails (case-insensitive)
    - Does NOT automatically log the user in
    """
    # 1. Validate Username
    u_ok, u_msg = validate_username(username)
    if not u_ok:
        return False, u_msg
    clean_username = normalize_username(username)

    # 2. Validate Email
    e_ok, e_msg = validate_email(email)
    if not e_ok:
        return False, e_msg
    clean_email = normalize_email(email)

    # 3. Validate Password
    p_ok, p_msg = validate_password(password)
    if not p_ok:
        return False, p_msg

    # 4. Validate Confirm Password
    if password != confirm_password:
        return False, "Passwords do not match."

    # 5. Validate Security Question & Answer
    q_ok, q_msg = validate_security_question(security_question)
    if not q_ok:
        return False, q_msg

    a_ok, a_msg = validate_security_answer(security_answer)
    if not a_msg:
        pass
    if not a_ok:
        return False, a_msg
    clean_answer = normalize_security_answer(security_answer)

    # 6. Database duplicate checks and atomic insertion
    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()

            cursor.execute("SELECT id FROM users WHERE LOWER(username) = LOWER(?)", (clean_username,))
            if cursor.fetchone():
                return False, "Username is already registered. Please choose another username."

            cursor.execute("SELECT id FROM users WHERE LOWER(email) = LOWER(?)", (clean_email,))
            if cursor.fetchone():
                return False, "Email address is already registered. Please sign in or use another email."

            pwd_hash = hash_text(password)
            ans_hash = hash_text(clean_answer)
            is_ver_val = 1 if auto_verify else 0

            cursor.execute(
                """
                INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, account_status, is_verified)
                VALUES (?, ?, ?, ?, ?, 'ACTIVE', ?)
                """,
                (clean_username, clean_email, pwd_hash, security_question, ans_hash, is_ver_val),
            )
            conn.commit()
            return True, "Account created successfully! Please verify your email to activate your account."

    except sqlite3.IntegrityError as e:
        err_str = str(e).lower()
        if "username" in err_str:
            return False, "Username is already registered. Please choose another username."
        elif "email" in err_str:
            return False, "Email address is already registered. Please sign in."
        return False, "Registration conflict: Username or Email is already registered."
    except Exception as e:
        return False, f"An unexpected error occurred during registration: {str(e)}"


# ==============================================================================
# 9. AUTHENTICATION & ACCOUNT LOCKOUT POLICY
# ==============================================================================
def authenticate_user(
    identifier: str, password: str, db_path: Optional[str] = None
) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Authenticates a user via Username OR Email:
    - Rejects unverified accounts
    - Enforces 5 failed attempts = 15-minute lockout
    - Auto-unlocks after lockout duration
    - Locked accounts cannot authenticate even with correct password
    - Protects against account enumeration via generic error messages & timing check
    """
    if not identifier or len(identifier.strip()) == 0:
        return False, None, "Please enter your username or email address."

    if not password:
        return False, None, "Please enter your password."

    clean_id = identifier.strip()

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT id, username, email, password_hash, account_status, failed_attempts, locked_until, is_verified
                FROM users
                WHERE LOWER(email) = LOWER(?) OR LOWER(username) = LOWER(?)
                """,
                (clean_id, clean_id),
            )
            user_row = cursor.fetchone()

            if not user_row:
                bcrypt.checkpw(b"dummy_timing_padding", b"$2b$12$e8Yd8eI9Z3mXyF9mH7P0seE8F0KqB9L5U2V0O1I2K3M4P5Q6R7S8T")
                return False, None, "Invalid username/email or password."

            user = dict(user_row)
            now_utc = datetime.datetime.utcnow()

            # Check Lockout status
            is_currently_locked = False
            if user.get("account_status") == "LOCKED":
                locked_until_str = user.get("locked_until")
                if locked_until_str:
                    try:
                        locked_until_dt = datetime.datetime.fromisoformat(locked_until_str)
                        if now_utc < locked_until_dt:
                            is_currently_locked = True
                            remaining_minutes = max(1, int((locked_until_dt - now_utc).total_seconds() / 60))
                            return (
                                False,
                                None,
                                f"Account is temporarily locked due to excessive failed attempts. Try again in {remaining_minutes} minute(s).",
                            )
                        else:
                            cursor.execute(
                                "UPDATE users SET account_status = 'ACTIVE', failed_attempts = 0, locked_until = NULL WHERE id = ?",
                                (user["id"],),
                            )
                            conn.commit()
                            user["account_status"] = "ACTIVE"
                            user["failed_attempts"] = 0
                    except Exception:
                        is_currently_locked = True
                else:
                    is_currently_locked = True

            if is_currently_locked:
                return False, None, "Account is temporarily locked. Please contact support."

            # Check Email Verification Status
            if not user.get("is_verified", 0):
                return False, None, "Account email is not verified. Please verify your email before logging in."

            # Verify password against bcrypt hash
            if check_hash(password, user["password_hash"]):
                if user.get("failed_attempts", 0) > 0 or user.get("locked_until") is not None:
                    cursor.execute(
                        "UPDATE users SET failed_attempts = 0, locked_until = NULL WHERE id = ?",
                        (user["id"],),
                    )
                    conn.commit()

                user_info = {
                    "id": user["id"],
                    "username": user["username"],
                    "email": user["email"],
                    "account_status": user.get("account_status", "ACTIVE"),
                    "is_verified": user.get("is_verified", 1),
                    "role": "USER",
                }
                return True, user_info, "Authentication successful."
            else:
                new_attempts = user.get("failed_attempts", 0) + 1
                if new_attempts >= MAX_FAILED_LOGIN_ATTEMPTS:
                    lock_expiry = (now_utc + datetime.timedelta(minutes=LOCKOUT_DURATION_MINUTES)).isoformat()
                    cursor.execute(
                        "UPDATE users SET failed_attempts = ?, account_status = 'LOCKED', locked_until = ? WHERE id = ?",
                        (new_attempts, lock_expiry, user["id"]),
                    )
                    conn.commit()
                    return (
                        False,
                        None,
                        f"Account locked due to {MAX_FAILED_LOGIN_ATTEMPTS} failed attempts. Please wait {LOCKOUT_DURATION_MINUTES} minutes.",
                    )
                else:
                    cursor.execute(
                        "UPDATE users SET failed_attempts = ? WHERE id = ?",
                        (new_attempts, user["id"]),
                    )
                    conn.commit()
                    return False, None, "Invalid username/email or password."

    except Exception as e:
        return False, None, f"Authentication error: {str(e)}"


# ==============================================================================
# 10. SEPARATE ADMIN AUTHENTICATION
# ==============================================================================
def authenticate_admin(
    identifier: str, password: str, db_path: Optional[str] = None
) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """Dedicated admin authentication against admin_users table."""
    if not identifier or not password:
        return False, None, "Please provide admin credentials."

    clean_id = identifier.strip()

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT id, username, email, password_hash, role
                FROM admin_users
                WHERE LOWER(email) = LOWER(?) OR LOWER(username) = LOWER(?)
                """,
                (clean_id, clean_id),
            )
            admin_row = cursor.fetchone()

            if not admin_row:
                bcrypt.checkpw(b"dummy_timing_padding", b"$2b$12$e8Yd8eI9Z3mXyF9mH7P0seE8F0KqB9L5U2V0O1I2K3M4P5Q6R7S8T")
                return False, None, "Invalid admin credentials."

            admin_data = dict(admin_row)
            if check_hash(password, admin_data["password_hash"]):
                return (
                    True,
                    {
                        "id": admin_data["id"],
                        "username": admin_data["username"],
                        "email": admin_data["email"],
                        "role": "ADMIN",
                    },
                    "Admin authentication successful.",
                )
            else:
                return False, None, "Invalid admin credentials."
    except Exception as e:
        return False, None, f"Admin authentication error: {str(e)}"


def get_all_users_for_admin(db_path: Optional[str] = None) -> List[Dict[str, Any]]:
    """Retrieves user list excluding password hashes, security answers, and OTPs."""
    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT id, username, email, account_status, is_verified, created_at
                FROM users
                ORDER BY id ASC
                """
            )
            rows = cursor.fetchall()
            return [dict(r) for r in rows]
    except Exception:
        return []


# ==============================================================================
# 11. SECURITY-QUESTION PASSWORD RECOVERY
# ==============================================================================
def get_security_question(identifier: str, db_path: Optional[str] = None) -> Tuple[bool, Optional[str], str]:
    """Retrieves the security question for a specific account."""
    if not identifier or len(identifier.strip()) == 0:
        return False, None, "Please enter your username or email address."

    clean_id = identifier.strip()
    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT security_question FROM users
                WHERE LOWER(email) = LOWER(?) OR LOWER(username) = LOWER(?)
                """,
                (clean_id, clean_id),
            )
            row = cursor.fetchone()
            if row:
                return True, row[0], "Security question retrieved."
            else:
                return False, None, "Account not found with provided identifier."
    except Exception as e:
        return False, None, f"Database error: {str(e)}"


def reset_password_with_security_question(
    identifier: str,
    answer: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None,
) -> Tuple[bool, str]:
    """Resets password verifying that answer matches that specific user's hash."""
    if not identifier or not answer:
        return False, "Please provide the account identifier and security answer."

    p_ok, p_msg = validate_password(new_password)
    if not p_ok:
        return False, p_msg

    if new_password != confirm_password:
        return False, "New passwords do not match."

    clean_id = identifier.strip()
    clean_ans = normalize_security_answer(answer)

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(
                """
                SELECT id, security_answer_hash FROM users
                WHERE LOWER(email) = LOWER(?) OR LOWER(username) = LOWER(?)
                """,
                (clean_id, clean_id),
            )
            row = cursor.fetchone()
            if not row:
                return False, "Account not found."

            user_id = row["id"]
            stored_answer_hash = row["security_answer_hash"]

            if not check_hash(clean_ans, stored_answer_hash):
                return False, "Incorrect security answer."

            new_pwd_hash = hash_text(new_password)
            cursor.execute(
                """
                UPDATE users
                SET password_hash = ?, failed_attempts = 0, account_status = 'ACTIVE', locked_until = NULL
                WHERE id = ?
                """,
                (new_pwd_hash, user_id),
            )
            conn.commit()
            return True, "Password reset successfully! Please sign in with your new password."

    except Exception as e:
        return False, f"Password reset error: {str(e)}"


# ==============================================================================
# 12. GMAIL OTP PASSWORD RECOVERY
# ==============================================================================
def create_otp_request(email: str, db_path: Optional[str] = None) -> Tuple[bool, Optional[str], str]:
    """Generates a 6-digit OTP code with 5-minute expiry, bcrypt-hashed storage, and 60s cooldown."""
    if not email:
        return False, None, "Email address is required."

    clean_email = normalize_email(email)

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT id FROM users WHERE LOWER(email) = LOWER(?)", (clean_email,))
            if not cursor.fetchone():
                return False, None, "No account found registered with this email address."

            now_utc = datetime.datetime.utcnow()

            cursor.execute(
                """
                SELECT created_at FROM otp_records
                WHERE LOWER(email) = LOWER(?) AND used = 0
                ORDER BY id DESC LIMIT 1
                """,
                (clean_email,),
            )
            last_record = cursor.fetchone()
            if last_record:
                try:
                    last_created = datetime.datetime.fromisoformat(last_record[0])
                    if (now_utc - last_created).total_seconds() < OTP_RESEND_COOLDOWN_SECONDS:
                        wait_sec = int(OTP_RESEND_COOLDOWN_SECONDS - (now_utc - last_created).total_seconds())
                        return False, None, f"Please wait {wait_sec} seconds before requesting a new OTP."
                except Exception:
                    pass

            cursor.execute("UPDATE otp_records SET used = 2 WHERE LOWER(email) = LOWER(?) AND used = 0", (clean_email,))

            otp_code = generate_code_6digit()
            otp_hash = hash_text(otp_code)
            expires_at = (now_utc + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)).isoformat()
            created_at = now_utc.isoformat()

            cursor.execute(
                """
                INSERT INTO otp_records (email, otp_hash, attempts, expires_at, used, created_at)
                VALUES (?, ?, 0, ?, 0, ?)
                """,
                (clean_email, otp_hash, expires_at, created_at),
            )
            conn.commit()
            return True, otp_code, "OTP generated successfully."

    except Exception as e:
        return False, None, f"Error generating OTP: {str(e)}"


def send_otp_via_smtp(
    to_email: str,
    otp_code: str,
    smtp_user: Optional[str] = None,
    smtp_pass: Optional[str] = None,
) -> Tuple[bool, str]:
    """Sends OTP using Gmail SMTP (SSL 465 / TLS 587 fallback)."""
    subject = "Your Infosys Portal Password Reset OTP"
    body = f"""Hello,

Your One-Time Password (OTP) for resetting your Infosys Portal account password is:

{otp_code}

This code is valid for {OTP_EXPIRY_MINUTES} minutes and can only be used once.
If you did not request this, please ignore this email.

Best regards,
Infosys Portal Security Team
"""
    return _dispatch_gmail_smtp(to_email, subject, body, smtp_user, smtp_pass)


def verify_and_reset_password_with_otp(
    email: str,
    otp_code: str,
    new_password: str,
    confirm_password: str,
    db_path: Optional[str] = None,
) -> Tuple[bool, str]:
    """Verifies OTP and resets user password (single-use, 5m expiry, max 3 tries)."""
    if not email or not otp_code:
        return False, "Email address and OTP code are required."

    p_ok, p_msg = validate_password(new_password)
    if not p_ok:
        return False, p_msg

    if new_password != confirm_password:
        return False, "New passwords do not match."

    clean_email = normalize_email(email)
    clean_otp = otp_code.strip()

    if not clean_otp.isdigit() or len(clean_otp) != 6:
        return False, "OTP code must be a 6-digit number."

    try:
        with get_db(db_path) as conn:
            cursor = conn.cursor()
            now_utc = datetime.datetime.utcnow()

            cursor.execute(
                """
                SELECT id, otp_hash, attempts, expires_at, used
                FROM otp_records
                WHERE LOWER(email) = LOWER(?) AND used = 0
                ORDER BY id DESC LIMIT 1
                """,
                (clean_email,),
            )
            otp_record = cursor.fetchone()

            if not otp_record:
                return False, "No active OTP request found. Please request a new OTP."

            otp_id = otp_record["id"]
            stored_hash = otp_record["otp_hash"]
            attempts = otp_record["attempts"]
            expires_at_str = otp_record["expires_at"]

            if attempts >= OTP_MAX_VERIFY_ATTEMPTS:
                cursor.execute("UPDATE otp_records SET used = 2 WHERE id = ?", (otp_id,))
                conn.commit()
                return False, "Maximum verification attempts exceeded. Please request a new OTP."

            try:
                expires_at_dt = datetime.datetime.fromisoformat(expires_at_str)
                if now_utc > expires_at_dt:
                    cursor.execute("UPDATE otp_records SET used = 2 WHERE id = ?", (otp_id,))
                    conn.commit()
                    return False, "OTP has expired. Please request a new code."
            except Exception:
                return False, "Invalid OTP timestamp."

            if not check_hash(clean_otp, stored_hash):
                cursor.execute("UPDATE otp_records SET attempts = attempts + 1 WHERE id = ?", (otp_id,))
                conn.commit()
                rem = OTP_MAX_VERIFY_ATTEMPTS - (attempts + 1)
                return False, f"Incorrect OTP code. {rem} attempt(s) remaining."

            cursor.execute("UPDATE otp_records SET used = 1 WHERE id = ?", (otp_id,))

            new_hash = hash_text(new_password)
            cursor.execute(
                """
                UPDATE users
                SET password_hash = ?, failed_attempts = 0, account_status = 'ACTIVE', locked_until = NULL
                WHERE LOWER(email) = LOWER(?)
                """,
                (new_hash, clean_email),
            )
            conn.commit()
            return True, "Password reset successfully via OTP! Please sign in with your new password."

    except Exception as e:
        return False, f"OTP verification error: {str(e)}"


# ==============================================================================
# 13. JWT & SESSION MANAGEMENT
# ==============================================================================
def make_jwt(email: str, role: str = "USER", expires_in_hours: int = 2) -> str:
    """Issues a signed JWT access token."""
    secret = get_jwt_secret()
    payload = {
        "email": email,
        "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=expires_in_hours),
        "iat": datetime.datetime.utcnow(),
    }
    return jwt.encode(payload, secret, algorithm=JWT_ALGORITHM)


def verify_jwt(token: Optional[str]) -> Optional[Dict[str, Any]]:
    """Verifies JWT token signature and expiration."""
    if not token:
        return None
    try:
        secret = get_jwt_secret()
        return jwt.decode(token, secret, algorithms=[JWT_ALGORITHM])
    except (jwt.ExpiredSignatureError, jwt.InvalidTokenError, ValueError):
        return None


# ==============================================================================
# 14. COLAB SECRETS INSPECTION HELPER (SAFE)
# ==============================================================================
def check_colab_secrets_status() -> Dict[str, bool]:
    """
    Safely checks presence of required Colab Secrets without exposing values.
    Returns: dict of secret_name -> is_present (bool)
    """
    secret_names = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_PASSWORD", "EMAIL_ADDRESS"]
    status = {}
    try:
        from google.colab import userdata
        for name in secret_names:
            try:
                val = userdata.get(name)
                status[name] = bool(val and len(str(val).strip()) > 0)
            except Exception:
                status[name] = False
    except Exception:
        for name in secret_names:
            val = os.environ.get(name)
            status[name] = bool(val and len(str(val).strip()) > 0)
    return status


In [ ]:
%%writefile app.py
"""
app.py - Infosys Springboard Internship 7.0 (Batch 4) - Milestone 1
Complete Web Application: Enterprise Authentication & Franchise Analytics Hub.
Featuring:
- High-contrast inputs (pure white background with dark readable text)
- Live password criteria checklist with instant visual feedback (green/red indicators)
- Live password confirmation matching indicator
- Show/hide password toggle controls
- Clean native Streamlit feature cards (zero raw HTML display)
- Email verification workflow (Gmail SMTP / simulated code fallback)
- Full backend integration with Bcrypt, SQLite, JWT sessions, Account Lockout,
  Security-Question recovery, and Gmail OTP recovery.
"""

import os
from typing import Optional
import streamlit as st
import plotly.graph_objects as go
from auth_utils import (
    init_db,
    register_user,
    authenticate_user,
    authenticate_admin,
    get_all_users_for_admin,
    get_security_question,
    reset_password_with_security_question,
    create_otp_request,
    send_otp_via_smtp,
    verify_and_reset_password_with_otp,
    create_email_verification,
    send_verification_email,
    verify_email_account,
    make_jwt,
    verify_jwt,
    check_password_criteria,
    APPROVED_SECURITY_QUESTIONS,
    get_email_credentials,
)

# ------------------------------------------------------------------------------
# 1. PAGE SETUP & ENHANCED HIGH-CONTRAST DESIGN SYSTEM
# ------------------------------------------------------------------------------
st.set_page_config(
    page_title="Infosys Portal - Authentication Hub",
    page_icon="⚡",
    layout="wide",
    initial_sidebar_state="collapsed",
)

THEME = {
    "bg_dark": "#0b0f19",
    "card_dark": "rgba(22, 30, 49, 0.9)",
    "border_subtle": "rgba(255, 255, 255, 0.14)",
    "primary": "#4f46e5",
    "primary_hover": "#4338ca",
    "text_light": "#f8fafc",
    "text_muted": "#94a3b8",
    "input_bg": "#ffffff",
    "input_text": "#0f172a",
    "input_border": "#cbd5e1",
    "input_focus": "#4f46e5",
}

st.markdown(
    f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Inter:wght@400;500;600;700&display=swap');

        /* Clean Background with ambient tech gradient */
        html, body, .stApp {{
            background-color: {THEME['bg_dark']} !important;
            background-image: 
                radial-gradient(at 15% 15%, rgba(79, 70, 229, 0.16) 0px, transparent 55%),
                radial-gradient(at 85% 85%, rgba(6, 182, 212, 0.13) 0px, transparent 50%),
                radial-gradient(at 50% 50%, rgba(15, 23, 42, 0.85) 0px, transparent 100%) !important;
            background-attachment: fixed !important;
            font-family: 'Plus Jakarta Sans', 'Inter', sans-serif !important;
            color: {THEME['text_light']} !important;
        }}

        footer, div[data-testid="stDecoration"] {{
            visibility: hidden !important;
            display: none !important;
        }}

        header {{
            background: transparent !important;
        }}

        .block-container {{
            padding: 2.2rem 3rem !important;
            max-width: 1240px;
        }}

        /* Typography */
        h1, h2, h3, h4 {{
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            color: #ffffff !important;
            font-weight: 700 !important;
            letter-spacing: -0.02em !important;
        }}

        label p {{
            font-weight: 600 !important;
            color: #f1f5f9 !important;
            font-size: 14px !important;
            margin-bottom: 5px !important;
            letter-spacing: 0.01em !important;
        }}

        /* -------------------------------------------------------------
           INPUT FIELDS: CRISP WHITE BACKGROUND & SOLID DARK TEXT (WCAG AAA)
           ------------------------------------------------------------- */
        div[data-baseweb="base-input"] {{
            background-color: transparent !important;
            border: none !important;
        }}

        div[data-baseweb="input"], div[data-baseweb="select"] {{
            background-color: {THEME['input_bg']} !important;
            border: 1.5px solid {THEME['input_border']} !important;
            border-radius: 10px !important;
            box-shadow: 0 2px 6px rgba(0, 0, 0, 0.15) !important;
            transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
        }}

        div[data-baseweb="input"]:hover, div[data-baseweb="select"]:hover {{
            border-color: #94a3b8 !important;
        }}

        div[data-baseweb="input"]:focus-within, div[data-baseweb="select"]:focus-within {{
            border-color: {THEME['input_focus']} !important;
            box-shadow: 0 0 0 3px rgba(79, 70, 229, 0.28), 0 4px 12px rgba(0, 0, 0, 0.25) !important;
        }}

        /* Typed Characters: Solid Dark Slate with maximum readability */
        input {{
            color: {THEME['input_text']} !important;
            -webkit-text-fill-color: {THEME['input_text']} !important;
            font-family: 'Inter', sans-serif !important;
            font-size: 15px !important;
            font-weight: 500 !important;
            padding: 10px 14px !important;
        }}

        input::placeholder {{
            color: #64748b !important;
            -webkit-text-fill-color: #64748b !important;
            font-size: 14px !important;
            font-weight: 400 !important;
        }}

        div[data-baseweb="select"] span {{
            color: {THEME['input_text']} !important;
            -webkit-text-fill-color: {THEME['input_text']} !important;
            font-weight: 600 !important;
        }}

        /* Completely remove confusing browser-like helper prompts such as 'Press Enter to apply' */
        div[data-testid="InputInstructions"] {{
            display: none !important;
        }}

        /* Checkbox styling (Show/Hide password) */
        div[data-testid="stCheckbox"] label {{
            margin-top: 4px !important;
            margin-bottom: 8px !important;
            cursor: pointer !important;
        }}

        div[data-testid="stCheckbox"] label p {{
            font-size: 13.5px !important;
            font-weight: 500 !important;
            color: #cbd5e1 !important;
        }}

        /* Action Buttons */
        div[data-testid="stButton"] button {{
            background: linear-gradient(135deg, #4f46e5 0%, #3b82f6 100%) !important;
            color: #ffffff !important;
            border: 1px solid rgba(255, 255, 255, 0.15) !important;
            border-radius: 10px !important;
            font-family: 'Plus Jakarta Sans', sans-serif !important;
            font-weight: 700 !important;
            font-size: 14.5px !important;
            height: 48px !important;
            min-height: 48px !important;
            white-space: nowrap !important;
            display: flex !important;
            align-items: center !important;
            justify-content: center !important;
            padding: 0px 20px !important;
            box-shadow: 0 6px 20px rgba(79, 70, 229, 0.4) !important;
            width: 100%;
            transition: all 0.2s ease-in-out !important;
        }}

        div[data-testid="stButton"] button:hover {{
            background: linear-gradient(135deg, #4338ca 0%, #2563eb 100%) !important;
            transform: translateY(-2px) !important;
            box-shadow: 0 10px 28px rgba(79, 70, 229, 0.55) !important;
        }}

        div[data-testid="stButton"] button:active {{
            transform: translateY(1px) !important;
            box-shadow: 0 3px 10px rgba(79, 70, 229, 0.3) !important;
        }}

        /* Secondary action buttons */
        .secondary-btn button {{
            background: rgba(30, 41, 59, 0.8) !important;
            border: 1px solid rgba(255, 255, 255, 0.16) !important;
            color: #e2e8f0 !important;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.25) !important;
        }}

        .secondary-btn button:hover {{
            background: rgba(51, 65, 85, 0.9) !important;
            border-color: rgba(255, 255, 255, 0.3) !important;
            color: #ffffff !important;
        }}

        /* Panels and Cards */
        .auth-card {{
            background: {THEME['card_dark']};
            backdrop-filter: blur(16px);
            -webkit-backdrop-filter: blur(16px);
            border: 1px solid {THEME['border_subtle']};
            border-radius: 18px;
            padding: 30px 28px;
            box-shadow: 0 20px 45px rgba(0, 0, 0, 0.45);
        }}

        .badge-tag {{
            display: inline-flex;
            align-items: center;
            padding: 5px 12px;
            background: rgba(79, 70, 229, 0.18);
            border: 1px solid rgba(79, 70, 229, 0.4);
            border-radius: 20px;
            color: #a5b4fc;
            font-size: 12px;
            font-weight: 700;
            letter-spacing: 0.04em;
            text-transform: uppercase;
            margin-bottom: 12px;
        }}

        /* Live Password Criteria Checklist Styles */
        .criteria-box {{
            background: rgba(15, 23, 42, 0.65);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 10px;
            padding: 12px 14px;
            margin-top: 6px;
            margin-bottom: 12px;
        }}

        .crit-item {{
            font-size: 13px;
            margin: 3px 0;
            display: flex;
            align-items: center;
            gap: 8px;
            font-weight: 500;
        }}

        .crit-pass {{
            color: #10b981;
            font-weight: 600;
        }}

        .crit-fail {{
            color: #f87171;
        }}

        .crit-neutral {{
            color: #94a3b8;
        }}

        /* Streamlit Tabs */
        div[data-baseweb="tab-list"] {{
            background-color: rgba(15, 23, 42, 0.8) !important;
            padding: 5px !important;
            border-radius: 12px !important;
            border: 1px solid rgba(255, 255, 255, 0.12) !important;
            margin-bottom: 20px !important;
        }}

        div[data-baseweb="tab"] {{
            border-radius: 8px !important;
            color: #94a3b8 !important;
            font-weight: 600 !important;
            font-size: 14px !important;
            padding: 8px 16px !important;
        }}

        div[data-baseweb="tab"][aria-selected="true"] {{
            background-color: #4f46e5 !important;
            color: #ffffff !important;
            box-shadow: 0 4px 12px rgba(79, 70, 229, 0.45) !important;
        }}
    </style>
    """,
    unsafe_allow_html=True,
)

# ------------------------------------------------------------------------------
# 2. DATABASE INITIALIZATION
# ------------------------------------------------------------------------------
init_db()

# ------------------------------------------------------------------------------
# 3. SESSION STATE INITIALIZATION
# ------------------------------------------------------------------------------
default_state = {
    "page": "Login",
    "jwt_token": None,
    "current_user": None,
    "signup_flash_msg": None,
    "verify_email_target": None,
    "recovery_identifier": None,
    "recovery_sq": None,
    "otp_requested_email": None,
}

for k, v in default_state.items():
    if k not in st.session_state:
        st.session_state[k] = v


def navigate_to(page_name: str):
    """Safely transitions between views."""
    st.session_state.page = page_name
    st.rerun()


def logout():
    """Clears authentication credentials and redirects to Login."""
    st.session_state.jwt_token = None
    st.session_state.current_user = None
    st.session_state.page = "Login"
    st.rerun()


def render_live_password_checklist(password: str, confirm_password: Optional[str] = None):
    """
    Renders live checklist for password criteria with green checks and red/neutral indicators.
    Updates immediately as the user types.
    """
    crit = check_password_criteria(password)
    has_typed = len(password) > 0

    def get_indicator(is_ok: bool) -> tuple:
        if is_ok:
            return ("✓", "crit-pass")
        elif has_typed:
            return ("✕", "crit-fail")
        else:
            return ("○", "crit-neutral")

    i_len, c_len = get_indicator(crit["min_length"])
    i_upper, c_upper = get_indicator(crit["uppercase"])
    i_lower, c_lower = get_indicator(crit["lowercase"])
    i_digit, c_digit = get_indicator(crit["digit"])
    i_spec, c_spec = get_indicator(crit["special"])

    st.markdown(
        f"""
        <div class="criteria-box">
            <div style="font-size: 12px; font-weight: 700; color: #cbd5e1; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 6px;">
                Password Policy Checklist:
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 4px;">
                <div class="crit-item {c_len}"><span>{i_len}</span> At least 8 characters</div>
                <div class="crit-item {c_upper}"><span>{i_upper}</span> At least 1 uppercase (A-Z)</div>
                <div class="crit-item {c_lower}"><span>{i_lower}</span> At least 1 lowercase (a-z)</div>
                <div class="crit-item {c_digit}"><span>{i_digit}</span> At least 1 number (0-9)</div>
                <div class="crit-item {c_spec}" style="grid-column: span 2;"><span>{i_spec}</span> At least 1 special character (!@#$%^&* etc.)</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    if confirm_password is not None:
        has_typed_confirm = len(confirm_password) > 0
        if has_typed_confirm:
            if password == confirm_password and len(password) > 0:
                st.markdown('<div style="color: #10b981; font-size: 13.5px; font-weight: 600; margin-top: -4px; margin-bottom: 8px;">✓ Passwords match exactly</div>', unsafe_allow_html=True)
            else:
                st.markdown('<div style="color: #f87171; font-size: 13.5px; font-weight: 600; margin-top: -4px; margin-bottom: 8px;">✕ Passwords do not match</div>', unsafe_allow_html=True)


# ------------------------------------------------------------------------------
# 4. PROTECTED ROUTES: USER & ADMIN DASHBOARDS
# ------------------------------------------------------------------------------
if st.session_state.jwt_token:
    payload = verify_jwt(st.session_state.jwt_token)
    if not payload:
        st.warning("⚠️ Session expired or invalid. Please sign in again.")
        logout()

    user_role = payload.get("role", "USER")
    user_email = payload.get("email")

    # ==========================================================================
    # ADMIN DASHBOARD
    # ==========================================================================
    if user_role == "ADMIN":
        st.markdown(
            f"""
            <div style="background: linear-gradient(135deg, rgba(30, 41, 59, 0.95), rgba(15, 23, 42, 0.95));
                        border: 1px solid rgba(79, 70, 229, 0.35); border-radius: 18px; padding: 24px 32px;
                        display: flex; justify-content: space-between; align-items: center; margin-bottom: 26px;
                        box-shadow: 0 20px 40px rgba(0, 0, 0, 0.45);">
                <div>
                    <span class="badge-tag" style="margin-bottom: 6px;">🛡️ Administrator Workspace</span>
                    <h1 style="margin: 0; font-size: 24px !important; color: #ffffff !important;">Infosys Administrator Control Center</h1>
                    <div style="color: #94a3b8; font-size: 13.5px; margin-top: 4px;">Role: System Administrator • Secure Session Active</div>
                </div>
                <div style="background: rgba(79, 70, 229, 0.2); border: 1px solid rgba(79, 70, 229, 0.5);
                            padding: 9px 20px; border-radius: 24px; font-weight: 700; color: #c7d2fe; font-size: 13.5px;">
                    🛡️ {user_email}
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        all_users = get_all_users_for_admin()
        active_count = sum(1 for u in all_users if u.get("account_status") == "ACTIVE")
        locked_count = sum(1 for u in all_users if u.get("account_status") == "LOCKED")

        c1, c2, c3, c4 = st.columns(4)
        c1.markdown(f'<div class="auth-card" style="padding:18px;text-align:center;"><div style="font-size:26px;">👥</div><div style="font-size:24px;font-weight:800;color:#f8fafc;">{len(all_users)}</div><div style="color:#94a3b8;font-size:12.5px;">Registered Users</div></div>', unsafe_allow_html=True)
        c2.markdown(f'<div class="auth-card" style="padding:18px;text-align:center;"><div style="font-size:26px;">✅</div><div style="font-size:24px;font-weight:800;color:#10b981;">{active_count}</div><div style="color:#94a3b8;font-size:12.5px;">Active Accounts</div></div>', unsafe_allow_html=True)
        c3.markdown(f'<div class="auth-card" style="padding:18px;text-align:center;"><div style="font-size:26px;">🔒</div><div style="font-size:24px;font-weight:800;color:#f43f5e;">{locked_count}</div><div style="color:#94a3b8;font-size:12.5px;">Locked Accounts</div></div>', unsafe_allow_html=True)
        c4.markdown(f'<div class="auth-card" style="padding:18px;text-align:center;"><div style="font-size:26px;">⚡</div><div style="font-size:24px;font-weight:800;color:#06b6d4;">HS256</div><div style="color:#94a3b8;font-size:12.5px;">JWT Token Auth</div></div>', unsafe_allow_html=True)

        st.markdown("<div style='height: 20px;'></div>", unsafe_allow_html=True)

        st.markdown(
            """
            <div class="auth-card" style="padding:22px 26px;margin-bottom:18px;">
                <h3 style="margin:0;font-size:18px;">📋 Registered Users Directory</h3>
                <p style="margin:4px 0 0;font-size:13px;color:#94a3b8;">
                    Protected administrative overview. Password hashes, security answers, and OTP records are strictly excluded.
                </p>
            </div>
            """,
            unsafe_allow_html=True,
        )

        if all_users:
            st.dataframe(all_users, use_container_width=True)
        else:
            st.info("No registered users found in the system yet.")

        st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
        if st.button("🚪 Sign Out from Admin Panel", use_container_width=False):
            logout()

    # ==========================================================================
    # REGULAR USER DASHBOARD
    # ==========================================================================
    else:
        current_uname = st.session_state.current_user.get("username", "Enterprise User") if st.session_state.current_user else "Enterprise User"

        st.markdown(
            f"""
            <div style="background: linear-gradient(135deg, rgba(30, 41, 59, 0.95), rgba(15, 23, 42, 0.95));
                        border: 1px solid rgba(79, 70, 229, 0.35); border-radius: 18px; padding: 24px 32px;
                        display: flex; justify-content: space-between; align-items: center; margin-bottom: 26px;
                        box-shadow: 0 20px 40px rgba(0, 0, 0, 0.45);">
                <div>
                    <span class="badge-tag" style="margin-bottom: 6px;">⚡ Franchise AI Platform</span>
                    <h1 style="margin: 0; font-size: 24px !important; color: #ffffff !important;">Intelligent Analytics Dashboard</h1>
                    <div style="color: #94a3b8; font-size: 13.5px; margin-top: 4px;">Welcome back, <strong>{current_uname}</strong> • Authenticated Session</div>
                </div>
                <div style="background: rgba(79, 70, 229, 0.2); border: 1px solid rgba(79, 70, 229, 0.5);
                            padding: 9px 20px; border-radius: 24px; font-weight: 700; color: #c7d2fe; font-size: 13.5px;">
                    👤 {user_email}
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [
            (c1, "📄", "Documents Indexed", "184"),
            (c2, "🔍", "Queries Executed", "76"),
            (c3, "📊", "Efficiency Score", "99.1%"),
            (c4, "🛡️", "Security Status", "Encrypted"),
        ]:
            col.markdown(
                f"""
                <div class="auth-card" style="padding:18px;text-align:center;">
                    <div style="font-size:26px;margin-bottom:4px;">{icon}</div>
                    <div style="font-size:24px;font-weight:800;color:#f8fafc;">{val}</div>
                    <div style="color:#94a3b8;font-size:12.5px;font-weight:600;">{lbl}</div>
                </div>
                """,
                unsafe_allow_html=True,
            )

        st.markdown("<div style='height: 20px;'></div>", unsafe_allow_html=True)

        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=96,
                number={"suffix": "%", "font": {"color": "#f8fafc", "size": 34, "family": "Plus Jakarta Sans"}},
                title={"text": "Franchise Health & Security Score", "font": {"color": "#cbd5e1", "size": 15, "family": "Plus Jakarta Sans"}},
                gauge={
                    "axis": {"range": [0, 100], "tickcolor": "#64748b"},
                    "bar": {"color": "#4f46e5", "thickness": 0.28},
                    "bgcolor": "rgba(30, 41, 59, 0.6)",
                    "borderwidth": 1,
                    "bordercolor": "rgba(255, 255, 255, 0.1)",
                    "steps": [
                        {"range": [0, 50], "color": "rgba(244, 63, 94, 0.25)"},
                        {"range": [50, 80], "color": "rgba(245, 158, 11, 0.25)"},
                        {"range": [80, 100], "color": "rgba(16, 185, 129, 0.25)"},
                    ],
                },
            )
        )
        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)",
            font={"color": "#f8fafc", "family": "Plus Jakarta Sans"},
            height=270,
            margin=dict(l=20, r=20, t=50, b=10),
        )
        st.plotly_chart(fig, use_container_width=True)

        st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
        if st.button("🚪 Sign Out", use_container_width=False):
            logout()

# ------------------------------------------------------------------------------
# 5. PUBLIC AUTHENTICATION: LOGIN, SIGNUP, VERIFY EMAIL, RECOVERY
# ------------------------------------------------------------------------------
else:
    # --------------------------------------------------------------------------
    # MAIN SIGN IN SCREEN: POLISHED SPLIT-SCREEN LAYOUT
    # --------------------------------------------------------------------------
    if st.session_state.page in ["Login", "AdminLogin"]:
        left_col, right_col = st.columns([1.1, 1], gap="large")

        # LEFT PANEL: NATIVE STREAMLIT FEATURE CARDS (NO RAW HTML CODE DISPLAYED)
        with left_col:
            st.markdown('<span class="badge-tag">⚡ INFOSYS SPRINGBOARD 7.0 • BATCH 4</span>', unsafe_allow_html=True)
            st.markdown("## Intelligent Franchise AI & Security Hub")
            st.caption("Enterprise-grade user authentication and access management system built with robust cryptographic controls.")
            
            st.markdown("<div style='height: 12px;'></div>", unsafe_allow_html=True)

            with st.container(border=True):
                st.markdown("#### 🛡️ Secure Password Hashing")
                st.write("Bcrypt cryptographic hashing with per-user salt rounds. Passwords and security answers are never stored in plaintext.")

            with st.container(border=True):
                st.markdown("#### ⚡ JWT Session Security")
                st.write("Stateless, cryptographically signed HS256 access tokens validating identity and role permissions with automatic expiry.")

            with st.container(border=True):
                st.markdown("#### 🔒 Account Lockout Protection")
                st.write("Automated 15-minute account lockout after 5 consecutive failed login attempts, defending against brute-force attacks.")

            with st.container(border=True):
                st.markdown("#### 📧 Two-Channel Password Recovery")
                st.write("Self-service credential recovery via approved Security Questions or 5-minute single-use Gmail OTP tokens.")

            st.markdown("---")
            c_s1, c_s2 = st.columns([1, 1])
            c_s1.caption("Milestone 1 • User Authentication Module")
            c_s2.markdown('<span style="color:#10b981;font-size:12.5px;font-weight:600;">● System Operational</span>', unsafe_allow_html=True)

        # RIGHT PANEL: LOGIN FORM WITH HIGH-CONTRAST INPUTS & SHOW/HIDE PASSWORD
        with right_col:
            st.markdown('<div class="auth-card">', unsafe_allow_html=True)

            if st.session_state.signup_flash_msg:
                st.success(f"✅ {st.session_state.signup_flash_msg}")
                st.session_state.signup_flash_msg = None

            tab_user, tab_admin = st.tabs(["👤 User Sign In", "🛡️ Admin Portal"])

            # TAB 1: USER SIGN IN
            with tab_user:
                st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)

                user_id = st.text_input(
                    "Username or Email",
                    placeholder="Enter registered username or email",
                    key="login_user_id",
                )

                show_user_pwd = st.checkbox("👁️ Show password", key="chk_show_user_pwd")
                user_pwd_type = "default" if show_user_pwd else "password"

                user_pwd = st.text_input(
                    "Password",
                    type=user_pwd_type,
                    placeholder="••••••••",
                    key="login_user_pwd",
                )

                st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)

                if st.button("Sign In →", key="btn_signin_user", use_container_width=True):
                    is_valid, user_data, err_msg = authenticate_user(user_id, user_pwd)
                    if is_valid and user_data:
                        try:
                            token = make_jwt(user_data["email"], role=user_data.get("role", "USER"))
                            st.session_state.current_user = user_data
                            st.session_state.jwt_token = token
                            st.rerun()
                        except ValueError as ve:
                            st.error(f"❌ Configuration Error: {str(ve)}")
                    else:
                        st.error(f"❌ {err_msg}")
                        if "not verified" in err_msg.lower():
                            st.session_state.verify_email_target = user_id.strip()
                            if st.button("Verify Email Now →", key="btn_goto_verify_unverified"):
                                navigate_to("VerifyEmail")

                st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)

                col_signup, col_forgot = st.columns(2)
                with col_signup:
                    st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                    if st.button("Create Account", key="btn_nav_signup", use_container_width=True):
                        navigate_to("Signup")
                    st.markdown('</div>', unsafe_allow_html=True)
                with col_forgot:
                    st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                    if st.button("Forgot Password?", key="btn_nav_forgot", use_container_width=True):
                        navigate_to("ForgotSQ")
                    st.markdown('</div>', unsafe_allow_html=True)

            # TAB 2: SEPARATE ADMIN PORTAL
            with tab_admin:
                st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
                st.caption("🛡️ Reserved for system administrators (Ordinary user credentials will be rejected).")

                admin_id = st.text_input(
                    "Admin Username or Email",
                    placeholder="Administrator or infosys@ai",
                    key="login_admin_id",
                )

                show_admin_pwd = st.checkbox("👁️ Show password", key="chk_show_admin_pwd")
                admin_pwd_type = "default" if show_admin_pwd else "password"

                admin_pwd = st.text_input(
                    "Admin Password",
                    type=admin_pwd_type,
                    placeholder="••••••••",
                    key="login_admin_pwd",
                )

                st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)

                if st.button("Admin Sign In →", key="btn_signin_admin", use_container_width=True):
                    is_valid, admin_data, err_msg = authenticate_admin(admin_id, admin_pwd)
                    if is_valid and admin_data:
                        try:
                            token = make_jwt(admin_data["email"], role="ADMIN")
                            st.session_state.current_user = admin_data
                            st.session_state.jwt_token = token
                            st.rerun()
                        except ValueError as ve:
                            st.error(f"❌ Configuration Error: {str(ve)}")
                    else:
                        st.error(f"❌ {err_msg}")

            st.markdown("</div>", unsafe_allow_html=True)

    # ==========================================================================
    # PAGE: SIGNUP WITH LIVE VALIDATION & EMAIL VERIFICATION DISPATCH
    # ==========================================================================
    elif st.session_state.page == "Signup":
        _, mid_col, _ = st.columns([1, 1.6, 1])
        with mid_col:
            st.markdown(
                """
                <div style="text-align: center; margin-bottom: 22px;">
                    <span class="badge-tag">⚡ Account Registration</span>
                    <h2 style="font-size: 26px; margin: 0 0 6px 0; color: #ffffff !important;">Join Infosys Platform</h2>
                    <p style="color: #94a3b8; font-size: 14px;">Create your enterprise credentials with strict security standards.</p>
                </div>
                """,
                unsafe_allow_html=True,
            )

            st.markdown('<div class="auth-card">', unsafe_allow_html=True)

            username = st.text_input("Username", placeholder="e.g. JaneDoe or SujalPatel", key="signup_username_input")
            email = st.text_input("Email address", placeholder="e.g. user@infosys.com", key="signup_email_input")

            show_signup_pwd = st.checkbox("👁️ Show passwords", key="chk_show_signup_pwd")
            signup_pwd_type = "default" if show_signup_pwd else "password"

            password = st.text_input("Password", type=signup_pwd_type, placeholder="Enter secure password", key="signup_pwd_input")
            confirm_password = st.text_input("Confirm password", type=signup_pwd_type, placeholder="Re-enter password", key="signup_conf_input")

            render_live_password_checklist(password, confirm_password)

            security_question = st.selectbox("Security Question", options=APPROVED_SECURITY_QUESTIONS, key="signup_sq_input")
            security_answer = st.text_input("Security Answer", placeholder="Answer to chosen security question", key="signup_sa_input")

            st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)

            if st.button("Complete Registration & Send Verification →", use_container_width=True):
                success, msg = register_user(
                    username=username,
                    email=email,
                    password=password,
                    confirm_password=confirm_password,
                    security_question=security_question,
                    security_answer=security_answer,
                )
                if success:
                    # Create email verification code and attempt SMTP send
                    ok_code, code_val, _ = create_email_verification(email)
                    st.session_state.verify_email_target = email.strip().lower()

                    smtp_user, smtp_pass = get_email_credentials()
                    if smtp_user and smtp_pass:
                        sent_ok, send_err = send_verification_email(email, code_val, smtp_user, smtp_pass)
                        if sent_ok:
                            st.session_state.signup_flash_msg = f"Verification code sent to {email}. Please check your inbox to activate your account."
                        else:
                            st.session_state.signup_flash_msg = f"Account registered, but verification email failed to deliver: {send_err}. Please use the verification screen to resend or enter your code."
                    else:
                        st.session_state.signup_flash_msg = "Account created. Please configure EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets to receive verification emails."

                    navigate_to("VerifyEmail")
                else:
                    st.error(f"❌ {msg}")

            st.markdown("<div style='height: 12px;'></div>", unsafe_allow_html=True)
            st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True):
                navigate_to("Login")
            st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)

    # ==========================================================================
    # PAGE: EMAIL VERIFICATION
    # ==========================================================================
    elif st.session_state.page == "VerifyEmail":
        _, mid_col, _ = st.columns([1, 1.6, 1])
        with mid_col:
            st.markdown(
                """
                <div style="text-align: center; margin-bottom: 22px;">
                    <span class="badge-tag">📧 Account Activation</span>
                    <h2 style="font-size: 26px; margin: 0 0 6px 0; color: #ffffff !important;">Verify Your Email Address</h2>
                    <p style="color: #94a3b8; font-size: 14px;">Enter the 6-digit verification code sent to your email address.</p>
                </div>
                """,
                unsafe_allow_html=True,
            )

            st.markdown('<div class="auth-card">', unsafe_allow_html=True)

            if st.session_state.signup_flash_msg:
                st.info(f"ℹ️ {st.session_state.signup_flash_msg}")
                st.session_state.signup_flash_msg = None

            target_email = st.text_input(
                "Registered Email Address",
                value=st.session_state.verify_email_target or "",
                placeholder="e.g. user@infosys.com",
                key="v_email_input",
            )

            v_code = st.text_input("Enter 6-Digit Verification Code", placeholder="6-digit code", max_chars=6, key="v_code_input")

            st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)

            if st.button("Verify Email & Activate Account →", use_container_width=True):
                ok, v_msg = verify_email_account(target_email, v_code)
                if ok:
                    st.session_state.signup_flash_msg = v_msg
                    st.session_state.verify_email_target = None
                    navigate_to("Login")
                else:
                    st.error(f"❌ {v_msg}")

            st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
            col_resend, col_back = st.columns(2)
            with col_resend:
                st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                if st.button("Resend Code", use_container_width=True):
                    ok_new, new_c, _ = create_email_verification(target_email)
                    smtp_user, smtp_pass = get_email_credentials()
                    if smtp_user and smtp_pass:
                        delivered, err_d = send_verification_email(target_email, new_c, smtp_user, smtp_pass)
                        if delivered:
                            st.success(f"✅ New code sent to {target_email}.")
                        else:
                            st.error(f"❌ Delivery error: {err_d}")
                    else:
                        st.error("❌ SMTP credentials missing in Colab Secrets. Cannot send verification email.")
                st.markdown('</div>', unsafe_allow_html=True)
            with col_back:
                st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                if st.button("← Back to Sign In", use_container_width=True):
                    navigate_to("Login")
                st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)

    # ==========================================================================
    # PAGE: FORGOT PASSWORD VIA SECURITY QUESTION
    # ==========================================================================
    elif st.session_state.page == "ForgotSQ":
        _, mid_col, _ = st.columns([1, 1.6, 1])
        with mid_col:
            st.markdown(
                """
                <div style="text-align: center; margin-bottom: 22px;">
                    <span class="badge-tag">🔒 Credential Recovery</span>
                    <h2 style="font-size: 26px; margin: 0 0 6px 0; color: #ffffff !important;">Security Question Verification</h2>
                    <p style="color: #94a3b8; font-size: 14px;">Answer your registered security question to reset your password.</p>
                </div>
                """,
                unsafe_allow_html=True,
            )

            st.markdown('<div class="auth-card">', unsafe_allow_html=True)

            if not st.session_state.recovery_identifier:
                ident = st.text_input("Registered Username or Email", placeholder="e.g. user@infosys.com", key="rec_sq_ident_input")

                st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
                col_find, col_sw = st.columns(2)
                with col_find:
                    if st.button("Find Account →", use_container_width=True):
                        ok, question, msg = get_security_question(ident)
                        if ok and question:
                            st.session_state.recovery_identifier = ident.strip()
                            st.session_state.recovery_sq = question
                            st.rerun()
                        else:
                            st.error("❌ Account not found. Please verify your username or email.")
                with col_sw:
                    st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                    if st.button("Use Gmail OTP →", use_container_width=True):
                        navigate_to("ForgotOTP")
                    st.markdown('</div>', unsafe_allow_html=True)

            else:
                st.info(f"❓ **Security Question:** {st.session_state.recovery_sq}")
                ans = st.text_input("Your Answer", placeholder="Enter your answer", key="rec_sq_ans_input")

                show_sq_pwd = st.checkbox("👁️ Show new passwords", key="chk_show_sq_pwd")
                sq_pwd_type = "default" if show_sq_pwd else "password"

                new_pwd = st.text_input("New Password", type=sq_pwd_type, placeholder="Min. 8 chars (upper, lower, digit, special)", key="rec_sq_npwd_input")
                confirm_pwd = st.text_input("Confirm New Password", type=sq_pwd_type, placeholder="Re-enter new password", key="rec_sq_cpwd_input")

                render_live_password_checklist(new_pwd, confirm_pwd)

                st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)
                if st.button("Reset Password →", use_container_width=True):
                    success, msg = reset_password_with_security_question(
                        identifier=st.session_state.recovery_identifier,
                        answer=ans,
                        new_password=new_pwd,
                        confirm_password=confirm_pwd,
                    )
                    if success:
                        st.session_state.recovery_identifier = None
                        st.session_state.recovery_sq = None
                        st.session_state.signup_flash_msg = msg
                        navigate_to("Login")
                    else:
                        st.error(f"❌ {msg}")

                st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
                st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                if st.button("← Choose Different Account", use_container_width=True):
                    st.session_state.recovery_identifier = None
                    st.session_state.recovery_sq = None
                    st.rerun()
                st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
            st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
            if st.button("← Cancel & Back to Login", use_container_width=True):
                st.session_state.recovery_identifier = None
                st.session_state.recovery_sq = None
                navigate_to("Login")
            st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)

    # ==========================================================================
    # PAGE: FORGOT PASSWORD VIA GMAIL OTP
    # ==========================================================================
    elif st.session_state.page == "ForgotOTP":
        _, mid_col, _ = st.columns([1, 1.6, 1])
        with mid_col:
            st.markdown(
                """
                <div style="text-align: center; margin-bottom: 22px;">
                    <span class="badge-tag">📧 One-Time Password</span>
                    <h2 style="font-size: 26px; margin: 0 0 6px 0; color: #ffffff !important;">Gmail OTP Recovery</h2>
                    <p style="color: #94a3b8; font-size: 14px;">Receive a secure 6-digit code via email with single-use verification.</p>
                </div>
                """,
                unsafe_allow_html=True,
            )

            st.markdown('<div class="auth-card">', unsafe_allow_html=True)

            if not st.session_state.otp_requested_email:
                target_email = st.text_input("Registered Email address", placeholder="e.g. user@infosys.com", key="rec_otp_email_input")

                st.markdown("<div style='height: 14px;'></div>", unsafe_allow_html=True)
                c_send, c_sw = st.columns(2)
                with c_send:
                    if st.button("Send 6-Digit OTP →", use_container_width=True):
                        ok, otp_code, msg = create_otp_request(target_email)
                        if ok and otp_code:
                            st.session_state.otp_requested_email = target_email.strip().lower()
                            smtp_user, smtp_pass = get_email_credentials()

                            if smtp_user and smtp_pass:
                                delivered, send_msg = send_otp_via_smtp(target_email, otp_code, smtp_user, smtp_pass)
                                if delivered:
                                    st.success(f"✅ OTP sent to {target_email}. Please check your inbox.")
                                else:
                                    st.error(f"❌ Failed to deliver OTP: {send_msg}")
                            else:
                                st.error("❌ SMTP credentials missing in Colab Secrets. Cannot send OTP.")
                            st.rerun()
                        else:
                            st.error(f"❌ {msg}")

                with c_sw:
                    st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                    if st.button("Use Security Question →", use_container_width=True):
                        navigate_to("ForgotSQ")
                    st.markdown('</div>', unsafe_allow_html=True)

            else:
                st.info(f"📧 Active OTP verification for: **{st.session_state.otp_requested_email}** (Valid for 5 minutes)")

                otp_input = st.text_input("Enter 6-digit OTP", placeholder="6-digit code", max_chars=6, key="rec_otp_code_input")

                show_otp_pwd = st.checkbox("👁️ Show new passwords", key="chk_show_otp_pwd")
                otp_pwd_type = "default" if show_otp_pwd else "password"

                new_pwd_otp = st.text_input("New Password", type=otp_pwd_type, placeholder="Min. 8 chars (upper, lower, digit, special)", key="rec_otp_npwd_input")
                conf_pwd_otp = st.text_input("Confirm New Password", type=otp_pwd_type, placeholder="Re-enter new password", key="rec_otp_cpwd_input")

                render_live_password_checklist(new_pwd_otp, conf_pwd_otp)

                st.markdown("<div style='height: 16px;'></div>", unsafe_allow_html=True)
                if st.button("Verify OTP & Reset Password →", use_container_width=True):
                    success, msg = verify_and_reset_password_with_otp(
                        email=st.session_state.otp_requested_email,
                        otp_code=otp_input,
                        new_password=new_pwd_otp,
                        confirm_password=conf_pwd_otp,
                    )
                    if success:
                        st.session_state.otp_requested_email = None
                        st.session_state.signup_flash_msg = msg
                        navigate_to("Login")
                    else:
                        st.error(f"❌ {msg}")

                st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
                st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
                if st.button("← Resend or Use Different Email", use_container_width=True):
                    st.session_state.otp_requested_email = None
                    st.session_state.otp_dev_code = None
                    st.rerun()
                st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
            st.markdown('<div class="secondary-btn">', unsafe_allow_html=True)
            if st.button("← Cancel & Back to Login", use_container_width=True):
                st.session_state.otp_requested_email = None
                st.session_state.otp_dev_code = None
                navigate_to("Login")
            st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)


In [ ]:
%%writefile test_auth.py
"""
test_auth.py - Automated Unit & Integration Tests for Milestone 1
Infosys Springboard Internship 7.0 (Batch 4) - User Authentication Module
Comprehensive test coverage for all Phase 1-5 requirements, live validation,
cross-account security mapping, OTP error handling, and account lockouts.
"""

import os
import sqlite3
import datetime
from unittest.mock import patch
import pytest
import jwt

# Configure test JWT_SECRET for test session
os.environ["JWT_SECRET"] = "test-session-secret-key-infosys-springboard-2026"

from auth_utils import (
    validate_password,
    validate_email,
    normalize_email,
    validate_username,
    normalize_username,
    validate_security_question,
    validate_security_answer,
    check_password_criteria,
    hash_text,
    check_hash,
    init_db,
    register_user,
    authenticate_user,
    authenticate_admin,
    get_all_users_for_admin,
    get_security_question,
    reset_password_with_security_question,
    create_otp_request,
    send_otp_via_smtp,
    verify_and_reset_password_with_otp,
    make_jwt,
    verify_jwt,
    get_jwt_secret,
    check_colab_secrets_status,
    create_email_verification,
    send_verification_email,
    verify_email_account,
    generate_code_6digit,
    MAX_FAILED_LOGIN_ATTEMPTS,
    LOCKOUT_DURATION_MINUTES,
    OTP_MAX_VERIFY_ATTEMPTS,
    VERIFICATION_EXPIRY_MINUTES,
)


# ==============================================================================
# TEST FIXTURES
# ==============================================================================
@pytest.fixture
def test_db(tmp_path):
    """Creates an isolated temporary SQLite database for test execution."""
    db_file = str(tmp_path / "test_auth_module.db")
    init_db(db_file, seed_admin=True)
    yield db_file
    if os.path.exists(db_file):
        try:
            os.remove(db_file)
        except Exception:
            pass


# ==============================================================================
# 1. LIVE PASSWORD CRITERIA HELPER TESTS
# ==============================================================================
class TestLivePasswordCriteriaHelper:
    """Tests the live criteria evaluation helper for instant UI checklist feedback."""

    def test_criteria_empty_password(self):
        crit = check_password_criteria("")
        assert not crit["min_length"]
        assert not crit["uppercase"]
        assert not crit["lowercase"]
        assert not crit["digit"]
        assert not crit["special"]
        assert not crit["all_passed"]

    def test_criteria_exactly_seven_characters(self):
        # Meets all criteria except min_length (7 chars)
        crit = check_password_criteria("Ab1!xyz")
        assert not crit["min_length"]
        assert crit["uppercase"]
        assert crit["lowercase"]
        assert crit["digit"]
        assert crit["special"]
        assert not crit["all_passed"]

    def test_criteria_exactly_eight_characters(self):
        crit = check_password_criteria("Abc123!@")
        assert crit["min_length"]
        assert crit["uppercase"]
        assert crit["lowercase"]
        assert crit["digit"]
        assert crit["special"]
        assert crit["all_passed"]

    def test_criteria_missing_uppercase(self):
        crit = check_password_criteria("lowercase123!@#")
        assert crit["min_length"]
        assert not crit["uppercase"]
        assert crit["lowercase"]
        assert crit["digit"]
        assert crit["special"]
        assert not crit["all_passed"]

    def test_criteria_missing_lowercase(self):
        crit = check_password_criteria("UPPERCASE123!@#")
        assert crit["min_length"]
        assert crit["uppercase"]
        assert not crit["lowercase"]
        assert crit["digit"]
        assert crit["special"]
        assert not crit["all_passed"]

    def test_criteria_missing_digit(self):
        crit = check_password_criteria("NoDigitsHere!@#")
        assert crit["min_length"]
        assert crit["uppercase"]
        assert crit["lowercase"]
        assert not crit["digit"]
        assert crit["special"]
        assert not crit["all_passed"]

    def test_criteria_missing_special(self):
        crit = check_password_criteria("NoSpecialChar123")
        assert crit["min_length"]
        assert crit["uppercase"]
        assert crit["lowercase"]
        assert crit["digit"]
        assert not crit["special"]
        assert not crit["all_passed"]

    def test_criteria_all_satisfied(self):
        crit = check_password_criteria("ValidP@ssw0rd")
        assert crit["min_length"]
        assert crit["uppercase"]
        assert crit["lowercase"]
        assert crit["digit"]
        assert crit["special"]
        assert crit["all_passed"]


# ==============================================================================
# 2. PASSWORD VALIDATION SUITE
# ==============================================================================
class TestPasswordValidation:
    """Tests password policy rules, edge cases, whitespace, and boundary conditions."""

    def test_empty_password(self):
        valid, msg = validate_password("")
        assert not valid
        assert "cannot be empty" in msg.lower()

    def test_whitespace_only_password(self):
        valid, msg = validate_password("       ")
        assert not valid
        assert "cannot be empty or whitespace only" in msg.lower()

    def test_length_below_eight(self):
        valid, msg = validate_password("Ab1!xyz")
        assert not valid
        assert "at least 8 characters" in msg.lower()

    def test_length_exactly_eight(self):
        valid, msg = validate_password("Abc123!@")
        assert valid
        assert msg == ""

    def test_longer_password(self):
        valid, msg = validate_password("SuperSecureLongPassword2026!#$")
        assert valid
        assert msg == ""

    def test_missing_uppercase(self):
        valid, msg = validate_password("lowercase123!@#")
        assert not valid
        assert "uppercase" in msg.lower()

    def test_missing_lowercase(self):
        valid, msg = validate_password("UPPERCASE123!@#")
        assert not valid
        assert "lowercase" in msg.lower()

    def test_missing_digit(self):
        valid, msg = validate_password("NoDigitsHere!@#")
        assert not valid
        assert "digit" in msg.lower()

    def test_missing_special_character(self):
        valid, msg = validate_password("NoSpecialChar123")
        assert not valid
        assert "special character" in msg.lower()

    def test_valid_password_all_conditions(self):
        valid, msg = validate_password("ValidP@ssw0rd")
        assert valid
        assert msg == ""

    def test_leading_trailing_spaces(self):
        valid, msg = validate_password(" ValidP@ssw0rd ")
        assert valid
        assert msg == ""

    def test_embedded_spaces(self):
        valid, msg = validate_password("Valid P@ss w0rd")
        assert valid
        assert msg == ""

    def test_unicode_and_unusual_characters(self):
        valid, msg = validate_password("VälidP@sswörd123!")
        assert valid
        assert msg == ""


# ==============================================================================
# 3. USERNAME & EMAIL VALIDATION TESTS
# ==============================================================================
class TestUsernameAndEmailValidation:
    """Tests email RFC syntax, 3-point min-length rules, and username constraints."""

    def test_empty_email(self):
        valid, msg = validate_email("")
        assert not valid
        assert "cannot be empty" in msg.lower()

    def test_whitespace_only_email(self):
        valid, msg = validate_email("     ")
        assert not valid
        assert "cannot be empty" in msg.lower()

    def test_email_missing_at_symbol(self):
        valid, msg = validate_email("username.domain.com")
        assert not valid
        assert "@" in msg

    def test_email_missing_local_part(self):
        valid, msg = validate_email("@domain.com")
        assert not valid
        assert "local part" in msg.lower()

    def test_email_missing_domain(self):
        valid, msg = validate_email("user@")
        assert not valid
        assert "domain" in msg.lower()

    def test_email_missing_final_dot(self):
        valid, msg = validate_email("user@domain")
        assert not valid
        assert "dot" in msg.lower()

    def test_email_multiple_at_symbols(self):
        valid, msg = validate_email("user@@domain.com")
        assert not valid
        assert "exactly one '@'" in msg

    def test_email_with_internal_spaces(self):
        valid, msg = validate_email("user name@domain.com")
        assert not valid
        assert "cannot contain spaces" in msg.lower()

    # Milestone 3-point rule tests
    def test_email_rule_min_two_letters_before_at(self):
        valid, msg = validate_email("a@domain.com")
        assert not valid
        assert "at least 2 characters" in msg.lower()

    def test_email_rule_min_two_letters_between_at_and_dot(self):
        valid, msg = validate_email("ab@c.com")
        assert not valid
        assert "at least 2 characters" in msg.lower()

    def test_email_rule_min_two_letters_after_final_dot(self):
        valid, msg = validate_email("ab@domain.c")
        assert not valid
        assert "at least 2 letters" in msg.lower()

    def test_email_valid_boundary_cases(self):
        valid_emails = [
            "ab@cd.ef",
            "jane.doe+test@infosys.com",
            "developer@sub.domain.co.uk",
            "user123@tech-corp.org",
            "contact@ai.global",
        ]
        for em in valid_emails:
            valid, msg = validate_email(em)
            assert valid, f"Expected '{em}' to be valid, got error: {msg}"

    def test_email_normalization(self):
        raw = "  Jane.Doe@Infosys.COM  "
        normalized = normalize_email(raw)
        assert normalized == "jane.doe@infosys.com"

    # Username tests
    def test_empty_username(self):
        valid, msg = validate_username("")
        assert not valid
        assert "cannot be empty" in msg.lower()

    def test_username_too_short(self):
        valid, msg = validate_username("ab")
        assert not valid
        assert "at least 3 characters" in msg.lower()

    def test_username_too_long(self):
        valid, msg = validate_username("a" * 51)
        assert not valid
        assert "exceed 50 characters" in msg.lower()

    def test_username_valid_formats(self):
        valid_usernames = ["JaneDoe", "john_smith", "Dev-Ops.2026", "Robert Brown"]
        for un in valid_usernames:
            valid, msg = validate_username(un)
            assert valid, f"Expected '{un}' to be valid, got: {msg}"

    def test_username_disallowed_characters(self):
        invalid_usernames = ["user@domain", "bad#user", "user!name", "<admin>"]
        for un in invalid_usernames:
            valid, msg = validate_username(un)
            assert not valid, f"Expected '{un}' to be invalid"


# ==============================================================================
# 4. SIGNUP FLOW & DATABASE INTEGRITY TESTS
# ==============================================================================
class TestSignupFlow:
    """Tests server-side signup validation, atomic insertion, hashing, and duplicate rejection."""

    def test_signup_successful(self, test_db):
        success, msg = register_user(
            username="AliceSmith",
            email="alice@infosys.com",
            password="SecurePassword123!",
            confirm_password="SecurePassword123!",
            security_question="What is your favourite city?",
            security_answer="London",
            db_path=test_db,
        )
        assert success
        assert "successfully" in msg.lower()

        with sqlite3.connect(test_db) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT username, email, password_hash, security_answer_hash, is_verified FROM users WHERE email='alice@infosys.com'")
            row = cursor.fetchone()
            assert row is not None
            un, em, pwd_hash, ans_hash, is_ver = row
            assert un == "AliceSmith"
            assert em == "alice@infosys.com"
            assert pwd_hash.startswith("$2b$")
            assert check_hash("SecurePassword123!", pwd_hash)
            assert ans_hash.startswith("$2b$")
            assert check_hash("london", ans_hash)
            assert is_ver == 0  # Requires verification before login

    def test_signup_password_confirmation_mismatch(self, test_db):
        success, msg = register_user(
            username="BobJones",
            email="bob@infosys.com",
            password="SecurePassword123!",
            confirm_password="DifferentPassword456!",
            security_question="What is your pet name?",
            security_answer="Buddy",
            db_path=test_db,
        )
        assert not success
        assert "do not match" in msg.lower()

    def test_signup_unapproved_security_question(self, test_db):
        success, msg = register_user(
            username="CharlieBrown",
            email="charlie@infosys.com",
            password="SecurePassword123!",
            confirm_password="SecurePassword123!",
            security_question="What is your lucky number?",
            security_answer="Seven",
            db_path=test_db,
        )
        assert not success
        assert "approved security question" in msg.lower()

    def test_signup_duplicate_username_rejected(self, test_db):
        s1, _ = register_user("SameUser", "first@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Cat", test_db)
        assert s1
        s2, m2 = register_user("SameUser", "second@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Dog", test_db)
        assert not s2
        assert "already registered" in m2.lower()
        s3, m3 = register_user("sameuser", "third@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Bird", test_db)
        assert not s3
        assert "already registered" in m3.lower()

    def test_signup_duplicate_email_rejected(self, test_db):
        s1, _ = register_user("UserOne", "dup@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Cat", test_db)
        assert s1
        s2, m2 = register_user("UserTwo", "DUP@INFOSYS.COM", "Password123!", "Password123!", "What is your pet name?", "Dog", test_db)
        assert not s2
        assert "already registered" in m2.lower()


# ==============================================================================
# 5. AUTHENTICATION & LOCKOUT PROTECTION TESTS
# ==============================================================================
class TestAuthenticationAndLockout:
    """Tests dual login, failed attempt tracking, 5-attempt lockout, and lock expiry."""

    @pytest.fixture(autouse=True)
    def setup_user(self, test_db):
        self.db = test_db
        register_user(
            username="JohnDoe",
            email="johndoe@infosys.com",
            password="SecretPassword123!",
            confirm_password="SecretPassword123!",
            security_question="What is your mother's maiden name?",
            security_answer="Williams",
            db_path=self.db,
            auto_verify=True,
        )

    def test_login_by_username_and_email(self):
        s1, u1, _ = authenticate_user("JohnDoe", "SecretPassword123!", db_path=self.db)
        assert s1 and u1["username"] == "JohnDoe"

        s2, u2, _ = authenticate_user("johndoe@infosys.com", "SecretPassword123!", db_path=self.db)
        assert s2 and u2["email"] == "johndoe@infosys.com"

    def test_incorrect_login_never_authenticates(self):
        # 1. Wrong password
        s1, u1, m1 = authenticate_user("JohnDoe", "WrongPass999!", db_path=self.db)
        assert not s1
        assert u1 is None
        assert "invalid username/email or password" in m1.lower()

        # 2. Non-existent user
        s2, u2, m2 = authenticate_user("NonExistentUser", "SecretPassword123!", db_path=self.db)
        assert not s2
        assert u2 is None
        assert "invalid username/email or password" in m2.lower()

        # 3. Empty fields
        s3, u3, _ = authenticate_user("", "SecretPassword123!", db_path=self.db)
        assert not s3
        assert u3 is None

        s4, u4, _ = authenticate_user("JohnDoe", "", db_path=self.db)
        assert not s4
        assert u4 is None

    def test_account_lockout_after_five_failed_attempts(self):
        # 1-4 failed attempts: account remains active
        for i in range(4):
            s, u, msg = authenticate_user("JohnDoe", "WrongPass123!", db_path=self.db)
            assert not s
            assert "invalid username/email or password" in msg.lower()

        # 5th failed attempt: triggers account lockout
        s5, u5, msg5 = authenticate_user("JohnDoe", "WrongPass123!", db_path=self.db)
        assert not s5
        assert "locked" in msg5.lower()

        # Verify in DB
        with sqlite3.connect(self.db) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT account_status, failed_attempts, locked_until FROM users WHERE username='JohnDoe'")
            status, attempts, locked_until = cursor.fetchone()
            assert status == "LOCKED"
            assert attempts == MAX_FAILED_LOGIN_ATTEMPTS
            assert locked_until is not None

        # Even with correct password, locked account must NOT authenticate
        s_corr, u_corr, msg_corr = authenticate_user("JohnDoe", "SecretPassword123!", db_path=self.db)
        assert not s_corr
        assert u_corr is None
        assert "locked" in msg_corr.lower()

    def test_lockout_auto_expiry(self):
        past_time = (datetime.datetime.utcnow() - datetime.timedelta(minutes=1)).isoformat()
        with sqlite3.connect(self.db) as conn:
            conn.execute(
                "UPDATE users SET account_status = 'LOCKED', failed_attempts = 5, locked_until = ? WHERE username='JohnDoe'",
                (past_time,),
            )

        # Login with correct password: auto-unlocks and succeeds
        success, user, _ = authenticate_user("JohnDoe", "SecretPassword123!", db_path=self.db)
        assert success
        assert user["account_status"] == "ACTIVE"


# ==============================================================================
# 5B. EMAIL VERIFICATION & UNVERIFIED ACCOUNT TESTS
# ==============================================================================
class TestEmailVerificationAndUnverifiedLogin:
    """Tests email verification workflow, unverified login prevention, OTP expiry, and attempt limits."""

    @pytest.fixture(autouse=True)
    def setup_user(self, test_db):
        self.db = test_db
        register_user(
            username="UnverifiedUser",
            email="unverified@infosys.com",
            password="SecretPassword123!",
            confirm_password="SecretPassword123!",
            security_question="What is your pet name?",
            security_answer="Fluffy",
            db_path=self.db,
            auto_verify=False,
        )

    def test_unverified_account_login_rejected(self):
        # User with is_verified = 0 must be rejected on login even with correct credentials
        success, user, msg = authenticate_user("UnverifiedUser", "SecretPassword123!", db_path=self.db)
        assert not success
        assert user is None
        assert "not verified" in msg.lower()

    def test_email_verification_code_generation(self):
        ok, code, msg = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok
        assert code is not None
        assert len(code) == 6
        assert code.isdigit()

    def test_email_verification_success_activates_account(self):
        ok, code, _ = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok

        # Verify email using generated code
        v_ok, v_msg = verify_email_account("unverified@infosys.com", code, db_path=self.db)
        assert v_ok
        assert "verified successfully" in v_msg.lower()

        # Account is now activated and can authenticate successfully
        auth_ok, user, _ = authenticate_user("UnverifiedUser", "SecretPassword123!", db_path=self.db)
        assert auth_ok
        assert user["is_verified"] == 1
        assert user["email"] == "unverified@infosys.com"

    def test_email_verification_wrong_code(self):
        ok, code, _ = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok

        # Submit incorrect verification code
        v_ok, v_msg = verify_email_account("unverified@infosys.com", "000000", db_path=self.db)
        assert not v_ok
        assert "incorrect verification code" in v_msg.lower()
        assert "2 attempt(s) remaining" in v_msg.lower()

    def test_email_verification_max_attempts_exceeded(self):
        ok, code, _ = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok

        # 3 failed verification attempts
        verify_email_account("unverified@infosys.com", "000001", db_path=self.db)
        verify_email_account("unverified@infosys.com", "000002", db_path=self.db)
        v_ok3, v_msg3 = verify_email_account("unverified@infosys.com", "000003", db_path=self.db)
        assert not v_ok3

        # Next attempt fails with max exceeded even with correct code
        v_ok4, v_msg4 = verify_email_account("unverified@infosys.com", code, db_path=self.db)
        assert not v_ok4
        assert "exceeded" in v_msg4.lower()

    def test_email_verification_expired_code(self):
        ok, code, _ = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok

        past_time = (datetime.datetime.utcnow() - datetime.timedelta(minutes=15)).isoformat()
        with sqlite3.connect(self.db) as conn:
            conn.execute("UPDATE email_verifications SET expires_at = ? WHERE email = 'unverified@infosys.com'", (past_time,))

        v_ok, v_msg = verify_email_account("unverified@infosys.com", code, db_path=self.db)
        assert not v_ok
        assert "expired" in v_msg.lower()

    def test_email_verification_single_use(self):
        ok, code, _ = create_email_verification("unverified@infosys.com", db_path=self.db)
        assert ok

        v_ok, _ = verify_email_account("unverified@infosys.com", code, db_path=self.db)
        assert v_ok

        # Reusing the already used code must fail
        v_reuse, m_reuse = verify_email_account("unverified@infosys.com", code, db_path=self.db)
        assert not v_reuse
        assert "no active verification request found" in m_reuse.lower()

    def test_send_verification_email_smtp_missing_credentials_fails_safely(self):
        sent, msg = send_verification_email("unverified@infosys.com", "123456", smtp_user="", smtp_pass="")
        assert not sent
        assert "credentials missing" in msg.lower()


# ==============================================================================
# 5C. CRYPTOGRAPHICALLY SECURE OTP GENERATION TESTS
# ==============================================================================
class TestSecureOtpGeneration:
    """Verifies that OTP generation uses secrets module, produces unpredictable 6-digit codes, and avoids hardcoded shortcuts."""

    def test_otp_format_and_range(self):
        code = generate_code_6digit()
        assert isinstance(code, str)
        assert len(code) == 6
        assert code.isdigit()
        val = int(code)
        assert 100000 <= val <= 999999

    def test_otp_unpredictability_and_non_sequential(self):
        codes = [generate_code_6digit() for _ in range(20)]
        unique_codes = set(codes)
        assert len(unique_codes) >= 18
        assert not all(c == "123456" for c in codes)
        diffs = [int(codes[i + 1]) - int(codes[i]) for i in range(len(codes) - 1)]
        assert not all(d == 1 for d in diffs)

    def test_both_workflows_generate_dynamic_otps(self, test_db):
        register_user("UserOne", "one@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Cat", test_db, auto_verify=True)
        # Workflow 1: Signup verification
        ok1, v_code, _ = create_email_verification("one@infosys.com", db_path=test_db)
        assert ok1
        assert len(v_code) == 6
        assert v_code.isdigit()
        assert 100000 <= int(v_code) <= 999999

        # Workflow 2: Forgot password OTP
        ok2, r_code, _ = create_otp_request("one@infosys.com", db_path=test_db)
        assert ok2
        assert len(r_code) == 6
        assert r_code.isdigit()
        assert 100000 <= int(r_code) <= 999999


# ==============================================================================
# 6. SECURITY-QUESTION RECOVERY & CROSS-ACCOUNT TESTS
# ==============================================================================
class TestSecurityQuestionRecovery:
    """Tests question retrieval, answer verification, password reset, and cross-account isolation."""

    @pytest.fixture(autouse=True)
    def setup_users(self, test_db):
        self.db = test_db
        register_user(
            username="SarahConnor",
            email="sarah@infosys.com",
            password="OriginalPassword123!",
            confirm_password="OriginalPassword123!",
            security_question="What is your pet name?",
            security_answer="Terminator",
            db_path=self.db,
            auto_verify=True,
        )
        register_user(
            username="JohnSmith",
            email="smith@infosys.com",
            password="SmithPassword123!",
            confirm_password="SmithPassword123!",
            security_question="What is your favourite city?",
            security_answer="Tokyo",
            db_path=self.db,
            auto_verify=True,
        )

    def test_get_security_question_accurate_per_account(self):
        ok1, q1, _ = get_security_question("SarahConnor", db_path=self.db)
        assert ok1
        assert q1 == "What is your pet name?"

        ok2, q2, _ = get_security_question("smith@infosys.com", db_path=self.db)
        assert ok2
        assert q2 == "What is your favourite city?"

    def test_cross_account_answer_rejected(self):
        # Attempt to reset SarahConnor using JohnSmith's answer 'Tokyo'
        success, msg = reset_password_with_security_question(
            identifier="SarahConnor",
            answer="Tokyo",
            new_password="NewSecurePass999!",
            confirm_password="NewSecurePass999!",
            db_path=self.db,
        )
        assert not success
        assert "incorrect security answer" in msg.lower()

    def test_recovery_correct_answer_updates_password(self):
        success, msg = reset_password_with_security_question(
            identifier="sarah@infosys.com",
            answer="terminator",
            new_password="BrandNewPassword999!",
            confirm_password="BrandNewPassword999!",
            db_path=self.db,
        )
        assert success
        assert "reset successfully" in msg.lower()

        # Old password must fail
        s_old, _, _ = authenticate_user("sarah@infosys.com", "OriginalPassword123!", db_path=self.db)
        assert not s_old

        # New password must succeed
        s_new, u_new, _ = authenticate_user("sarah@infosys.com", "BrandNewPassword999!", db_path=self.db)
        assert s_new
        assert u_new["email"] == "sarah@infosys.com"

    def test_recovery_password_policy_enforcement(self):
        success, msg = reset_password_with_security_question(
            identifier="SarahConnor",
            answer="terminator",
            new_password="weak",
            confirm_password="weak",
            db_path=self.db,
        )
        assert not success
        assert "at least 8 characters" in msg.lower()


# ==============================================================================
# 7. OTP RECOVERY, ERRORS & CROSS-ACCOUNT TESTS
# ==============================================================================
class TestOtpPasswordRecovery:
    """Tests OTP generation, 5-minute expiry, max 3 attempts limit, single-use, and cross-account rejection."""

    @pytest.fixture(autouse=True)
    def setup_users(self, test_db):
        self.db = test_db
        register_user(
            username="DavidMiller",
            email="david@infosys.com",
            password="OriginalPassword123!",
            confirm_password="OriginalPassword123!",
            security_question="What is your favourite city?",
            security_answer="Tokyo",
            db_path=self.db,
        )
        register_user(
            username="EmmaWatson",
            email="emma@infosys.com",
            password="EmmaPassword123!",
            confirm_password="EmmaPassword123!",
            security_question="What is your pet name?",
            security_answer="Bella",
            db_path=self.db,
        )

    def test_otp_generation_and_storage(self):
        ok, otp, msg = create_otp_request("david@infosys.com", db_path=self.db)
        assert ok
        assert len(otp) == 6
        assert otp.isdigit()

        with sqlite3.connect(self.db) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT otp_hash FROM otp_records WHERE email='david@infosys.com'")
            otp_hash = cursor.fetchone()[0]
            assert otp_hash != otp
            assert otp_hash.startswith("$2b$")
            assert check_hash(otp, otp_hash)

    def test_cross_account_otp_reset_rejected(self):
        # Generate OTP for David
        ok, david_otp, _ = create_otp_request("david@infosys.com", db_path=self.db)
        assert ok

        # Attempt to reset Emma's password using David's OTP
        success, msg = verify_and_reset_password_with_otp(
            email="emma@infosys.com",
            otp_code=david_otp,
            new_password="NewEmmaPassword123!",
            confirm_password="NewEmmaPassword123!",
            db_path=self.db,
        )
        assert not success
        assert "no active otp request found" in msg.lower()

    def test_otp_smtp_missing_credentials_fails_safely(self):
        # send_otp_via_smtp with empty credentials must fail safely without pretending it was sent
        delivered, msg = send_otp_via_smtp("david@infosys.com", "123456", smtp_user="", smtp_pass="")
        assert not delivered
        assert "missing" in msg.lower()

    def test_otp_verification_and_single_use(self):
        ok, otp, _ = create_otp_request("david@infosys.com", db_path=self.db)
        assert ok

        success, msg = verify_and_reset_password_with_otp(
            email="david@infosys.com",
            otp_code=otp,
            new_password="NewOtpPassword456!",
            confirm_password="NewOtpPassword456!",
            db_path=self.db,
        )
        assert success
        assert "reset successfully" in msg.lower()

        # Reusing the same OTP must fail (Single-use enforcement)
        s_reuse, m_reuse = verify_and_reset_password_with_otp(
            email="david@infosys.com",
            otp_code=otp,
            new_password="AnotherPassword789!",
            confirm_password="AnotherPassword789!",
            db_path=self.db,
        )
        assert not s_reuse

    def test_otp_max_attempt_limit(self):
        ok, otp, _ = create_otp_request("david@infosys.com", db_path=self.db)
        assert ok

        # Fail 3 consecutive attempts
        for i in range(OTP_MAX_VERIFY_ATTEMPTS):
            s, m = verify_and_reset_password_with_otp(
                email="david@infosys.com",
                otp_code="000000",
                new_password="NewPassword123!",
                confirm_password="NewPassword123!",
                db_path=self.db,
            )
            assert not s

        # 4th attempt: blocked
        s4, m4 = verify_and_reset_password_with_otp(
            email="david@infosys.com",
            otp_code=otp,
            new_password="NewPassword123!",
            confirm_password="NewPassword123!",
            db_path=self.db,
        )
        assert not s4
        assert "exceeded" in m4.lower()


# ==============================================================================
# 8. SEPARATE ADMIN AUTHENTICATION TESTS
# ==============================================================================
class TestAdminAuthentication:
    """Tests dedicated admin authentication, invalid credentials, and sanitized user listing."""

    def test_admin_authentication_success(self, test_db):
        ok, admin_info, msg = authenticate_admin("infosys@ai", "Admin@123", db_path=test_db)
        assert ok
        assert admin_info["role"] == "ADMIN"
        assert admin_info["email"] == "infosys@ai"

    def test_admin_authentication_wrong_credentials(self, test_db):
        ok, _, msg = authenticate_admin("infosys@ai", "WrongAdminPass", db_path=test_db)
        assert not ok
        assert "invalid admin credentials" in msg.lower()

    def test_ordinary_user_cannot_login_as_admin(self, test_db):
        register_user("RegularUser", "reg@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Dog", test_db)
        ok, _, _ = authenticate_admin("reg@infosys.com", "Password123!", db_path=test_db)
        assert not ok

    def test_admin_user_listing_excludes_sensitive_data(self, test_db):
        register_user("UserAlpha", "alpha@infosys.com", "Password123!", "Password123!", "What is your pet name?", "Cat", test_db)
        users = get_all_users_for_admin(db_path=test_db)
        assert len(users) >= 1
        for u in users:
            assert "password_hash" not in u
            assert "security_answer_hash" not in u
            assert "otp_hash" not in u
            assert "username" in u
            assert "email" in u
            assert "account_status" in u


# ==============================================================================
# 9. JWT & SESSION MANAGEMENT TESTS
# ==============================================================================
class TestJwtSessions:
    """Tests JWT creation, payload structure, expiration, tampering, and missing secret behavior."""

    def test_jwt_issuance_and_verification(self):
        token = make_jwt("testuser@infosys.com", role="USER")
        payload = verify_jwt(token)
        assert payload is not None
        assert payload["email"] == "testuser@infosys.com"
        assert payload["role"] == "USER"

    def test_jwt_tampering_rejected(self):
        token = make_jwt("testuser@infosys.com", role="USER")
        tampered_token = token[:-5] + "ABCDE"
        payload = verify_jwt(tampered_token)
        assert payload is None

    def test_jwt_expired_token_rejected(self):
        expired_token = make_jwt("testuser@infosys.com", expires_in_hours=-1)
        payload = verify_jwt(expired_token)
        assert payload is None

    def test_missing_jwt_secret_raises_value_error(self):
        # Temporarily remove JWT_SECRET from environment
        old_val = os.environ.pop("JWT_SECRET", None)
        try:
            with pytest.raises(ValueError) as excinfo:
                get_jwt_secret()
            assert "missing" in str(excinfo.value).lower()
        finally:
            if old_val is not None:
                os.environ["JWT_SECRET"] = old_val


# ==============================================================================
# 10. COLAB SECRETS STATUS CHECK (SAFE)
# ==============================================================================
class TestColabSecretsSafety:
    """Verifies that secrets status helper reports boolean presence without leaking values."""

    def test_secrets_status_returns_booleans(self):
        status = check_colab_secrets_status()
        assert isinstance(status, dict)
        for key in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_PASSWORD", "EMAIL_ADDRESS"]:
            assert key in status
            assert isinstance(status[key], bool)


In [ ]:
# ------------------------------------------------------------------------------
# 3. Execute Automated Unit & Integration Tests (77 Test Cases)
# ------------------------------------------------------------------------------
!pytest -v test_auth.py


In [ ]:
# ------------------------------------------------------------------------------
# 4. Launch Streamlit Application via Ngrok Tunnel
# ------------------------------------------------------------------------------
import os, time, subprocess
from pyngrok import ngrok

# Retrieve NGROK_AUTHTOKEN securely from Colab Secrets or Environment
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    pass
if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if token:
    clean_token = str(token).strip()
    ngrok.set_auth_token(clean_token)
    print('✅ Ngrok authentication token configured successfully.')
else:
    print('⚠️ NGROK_AUTHTOKEN not configured. Tunnel may be rate-limited.')

# Kill any previous streamlit sessions or active ngrok tunnels
!pkill -f streamlit || true
try:
    ngrok.kill()
except Exception:
    pass

# Start Streamlit server in background with headless flag
env = os.environ.copy()
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)
print('⏳ Starting Streamlit server...')
time.sleep(4)

# Establish Ngrok public tunnel
try:
    tunnel = ngrok.connect(8501)
    public_url = tunnel.public_url
    print('=' * 65)
    print(f'🚀 Infosys Portal Live URL: {public_url}')
    print('=' * 65)
    print('App is running live! Press Stop [■] in Google Colab to shut down.')
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('\n🛑 Shutting down tunnel and server...')
    ngrok.kill()
    process.terminate()
    print('✅ Server shut down cleanly.')
